# Step 10 — Recent-price reconstruction sensitivity
Run **Runtime → Run all**, then **save to GitHub as 09_fresh_price_sensitivity.ipynb**.

**Question:** does the association survive a reconstruction that excludes old prices and handles simultaneous trades consistently?

This is a measurement sensitivity, not a replacement validated predictor. The outcome, six-month benchmark, prior-day cutoff, and regression specifications stay fixed. The data below are small, checksum-verified derived snapshots, so this notebook needs no live data downloads. Raw transaction records are not included.


### Verified local run — October 19, 2026
The 48-hour rule retains **25/37** meetings; the 24-hour rule retains **21/37**. Benchmark-adjusted fresh central-band coefficients are respectively **0.311** (95% CI **0.001 to 0.621**) and **0.288** (**−0.092 to 0.667**). Both are positive, but the stricter result is less precise. Fresh and authors’ same-band estimates are very close. The evidence remains provisional: this exercise changes both sample coverage and the definition of dispersion. Rerun the cells below to reproduce these results.


In [ ]:
import io, json, hashlib, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import isotonic_regression
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)
if __import__('importlib').util.find_spec('statsmodels') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
import statsmodels.api as sm
OUT = Path('research_outputs/fresh_price_sensitivity')
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
SNAPSHOTS = {'meetings_csv': 'meeting_date,event_key,observation_date,authors_adjusted_mean_rate_pct,kalshi_sigma_bps,ust2y_signed_bps,abs_ust2y_bps,authors_daily_volume,probability_sum,bins,mpu_observation_date,mpu6m_bps,rv5_bps,mean_abs_daily_revision_bps,revision_level_observations,revision_window_complete,revision_calendar_review,window_zero_volume_days,sample_A,sample_B,sample_revision\n2022-01-26,FED-22JAN,2022-01-25,0.1326530612244898,4.306618116360679,-0.19,0.19,520.0,1.0,2.0,2022-01-25,41.712731463956,1.0518126595963397,0.3571428571428562,6,True,False,1,True,True,True\n2022-03-16,FED-22MAR,2022-03-15,0.3877551020408163,7.46800059607369,10.89999999999999,10.89999999999999,16578.0,1.0,4.0,2022-03-15,83.62845231731009,1.5931117342853054,0.4591836734693871,6,True,False,0,True,True,True\n2022-05-04,FED-22MAY,2022-05-03,0.8698979591836735,7.969642526435361,-4.70000000000001,4.70000000000001,14607.0,1.0,5.0,2022-05-03,102.297221047338,3.286759879266623,1.1224489795918402,6,True,False,0,True,True,True\n2022-06-15,FED-22JUN,2022-06-14,1.5841836734693877,13.216156986723067,5.04999999999999,5.04999999999999,78419.0,0.9999999999999999,8.0,2022-06-14,123.957247437196,15.62174580856615,4.0816326530612175,6,True,False,0,True,True,True\n2022-07-27,FED-22JUL,2022-07-26,2.392857142857143,8.183170883849671,0.20000000000002,0.20000000000002,29117.0,1.0,12.0,2022-07-26,108.859006956766,0.9197834886387686,0.35714285714285554,6,True,False,0,True,True,True\n2022-09-21,FED-22SEP,2022-09-20,3.137755102040816,6.558653128664452,9.38999999999996,9.38999999999996,75777.0,1.0,15.0,2022-09-20,92.9415372666587,2.1343368023828293,0.7142857142857111,6,True,False,0,True,True,True\n2022-11-02,FED-22NOV,2022-11-01,3.8673469387755097,5.594824540678823,-8.72000000000001,8.72000000000001,19163.0,1.0,17.0,2022-11-01,90.4278721308765,1.1119640162094546,0.35714285714287825,6,True,False,0,True,True,True\n2022-12-14,FED-22DEC,2022-12-13,4.341836734693878,8.479984764589085,8.579999999999979,8.579999999999979,88795.0,1.0,20.0,2022-12-13,72.31035143181501,3.893963653692261,1.2755102040816382,6,True,False,0,True,True,True\n2023-02-01,FED-23FEB,2023-01-31,4.625,3.5714285714285254,1.96999999999993,1.96999999999993,9035.0,1.0,16.0,2023-01-31,53.7254465737449,0.7653061224489193,0.2551020408163026,6,True,False,0,True,True,True\n2023-03-22,FED-23MAR,2023-03-21,4.839285714285714,10.714285714285671,-8.40999999999994,8.40999999999994,91020.0,0.9999999999999999,16.0,2023-03-21,152.481969536107,9.903086195807104,3.4183673469388167,6,True,False,0,True,True,True\n2023-05-03,FED-23MAY,2023-05-02,5.10969387755102,5.993540879831971,0.0,0.0,86682.0,1.0,16.0,2023-05-02,125.823612409271,2.119036699723989,0.7653061224489761,6,True,False,0,True,True,True\n2023-06-14,FED-23JUN,2023-06-13,5.135204081632653,4.946612099404399,11.77500000000009,11.77500000000009,99329.0,1.0,16.0,2023-06-13,99.5817222388473,3.86881400206713,1.2244897959183845,6,True,False,0,True,True,True\n2023-07-26,FED-23JUL,2023-07-25,5.377551020408163,2.512463724947986,0.41999999999999,0.41999999999999,2267.0,1.0,14.0,2023-07-25,82.1720108746314,0.8836993916167045,0.3061224489795677,6,True,False,1,True,True,True\n2023-09-20,FED-23SEP,2023-09-19,5.377551020408163,2.512463724947986,6.524999999999979,6.524999999999979,8716.0,1.0,16.0,2023-09-19,76.8537512311044,0.3607687659115826,0.10204081632655289,6,True,False,0,True,True,True\n2023-11-01,FED-23NOV,2023-10-31,5.375,0.0,-1.4600000000000601,1.4600000000000601,16534.0,1.0,15.0,2023-10-31,95.9057155444613,0.4418496958082866,0.15306122448976112,6,True,False,0,True,True,True\n2024-01-31,FED-24JAN,2024-01-30,5.380102040816325,3.534797566466996,1.44000000000001,1.44000000000001,2000.0,1.0,7.0,2024-01-30,78.9649626862937,0.36076876591126106,0.10204081632646193,6,True,False,1,True,True,True\n2024-03-20,FED-24MAR,2024-03-19,5.375,0.0,-2.7699999999999303,2.7699999999999303,11327.0,1.0,8.0,2024-03-19,71.4013628643701,0.0,0.0,6,True,False,2,True,True,True\n2024-05-01,FED-24MAY,2024-04-30,5.367346938775511,4.306618116360679,-1.04000000000006,1.04000000000006,350.0,1.0,8.0,2024-04-30,69.2304670691913,0.2551020408162685,0.051020408163253705,6,True,False,1,True,True,True\n2024-06-12,FED-24JUN,2024-06-11,5.369897959183672,3.534797566466996,1.68000000000008,1.68000000000008,17962.0,1.0,8.0,2024-06-11,64.6344164650726,0.0,0.0,6,True,False,0,True,True,True\n2024-07-31,FED-24JUL,2024-07-30,5.372448979591836,2.512463724947986,1.02999999999999,1.02999999999999,1046.0,1.0,8.0,2024-07-30,80.8933725788909,0.8836993916167044,0.3061224489795677,6,True,False,0,True,True,True\n2024-09-18,FED-24SEP,2024-09-17,4.99234693877551,12.476550265956613,-8.58000000000003,8.58000000000003,15270.0,1.0,9.0,2024-09-17,110.877098341297,8.468465649335707,3.571428571428612,6,True,False,0,True,True,True\n2024-11-07,FED-24NOV,2024-11-06,4.63265306122449,4.306618116360679,1.25000000000001,1.25000000000001,24257.0,1.0,10.0,2024-11-06,74.13029258910811,2.840696103484845,1.0204081632653357,6,True,False,0,True,True,True\n2024-12-18,FED-24DEC,2024-12-17,4.375,9.449111825230666,9.08000000000006,9.08000000000006,6736.0,1.0,11.0,2024-12-17,64.20995767151399,3.749218994056069,1.3265306122449716,6,True,False,0,True,True,True\n2025-01-29,FED-25JAN,2025-01-28,4.405612244897959,8.195091022555575,2.89999999999999,2.89999999999999,1051.0,1.0,11.0,2025-01-28,57.4167178456226,0.8067034847368637,0.30612244897959046,6,True,False,1,True,True,True\n2025-03-19,FED-25MAR,2025-03-18,4.377551020408164,2.512463724947986,-5.10000000000001,5.10000000000001,155.0,1.0,10.0,2025-03-18,65.5754801694924,0.5704255044642549,0.15306122448980658,6,True,False,0,True,True,True\n2025-05-07,FED-25MAY,2025-05-06,4.369897959183674,3.534797566466996,-3.10000000000001,3.10000000000001,91776.0,1.0,12.0,2025-05-06,81.4212909326843,1.8217929664650365,0.6632653061224574,6,True,False,0,True,True,True\n2025-06-18,FED-25JUN,2025-06-17,4.372448979591836,2.512463724947986,-2.4,2.4,112904.0,1.0,12.0,2025-06-17,69.5388581359056,0.2551020408162685,0.051020408163253705,6,True,False,0,True,True,True\n2025-07-30,FED-25JUL,2025-07-29,4.369897959183674,3.534797566466996,-1.00000000000002,1.00000000000002,203254.0,1.0,12.0,2025-07-29,60.5183004277037,0.765306122449033,0.25510204081634813,6,True,False,0,True,True,True\n2025-09-17,FED-25SEP,2025-09-16,4.11734693877551,6.637557056764108,-5.55000000000003,5.55000000000003,165524.0,1.0,12.0,2025-09-16,57.9748626947689,1.5306122448979143,0.5102040816326052,6,True,False,0,True,True,True\n2025-10-29,FED-25OCT,2025-10-28,3.875,3.5714285714285254,0.70000000000001,0.70000000000001,26302.0,1.0,12.0,2025-10-28,53.3512663118075,1.325549087425144,0.35714285714284416,6,True,False,0,True,True,True\n2025-12-10,FED-25DEC,2025-12-09,3.63265306122449,4.306618116360679,-2.20000000000002,2.20000000000002,15672.0,1.0,12.0,2025-12-09,53.9605761119063,1.3972514222070709,0.5102040816326507,6,True,False,0,True,True,True\n2026-01-28,FED-26JAN,2026-01-27,3.622448979591837,2.512463724947986,0.15,0.15,128650.0,1.0,12.0,2026-01-27,49.3759067140691,1.4203480517424378,0.56122448979595,6,True,False,0,True,True,True\n2026-03-18,FED-26MAR,2026-03-17,3.625,0.0,-0.8999999999999799,0.8999999999999799,156318.0,1.0,12.0,2026-03-17,64.83844435623199,0.8460777526416798,0.25510204081632537,6,True,False,0,True,True,True\n2026-04-29,FED-26APR,2026-04-28,3.625,0.0,2.09999999999999,2.09999999999999,352026.43,1.0,12.0,2026-04-28,65.9550666821221,0.441849695808385,0.15306122448979523,6,True,False,0,True,True,True\n2026-06-17,FED-26JUN,2026-06-16,3.625,0.0,7.10000000000006,7.10000000000006,171349.87,1.0,12.0,2026-06-16,54.1634920197275,0.0,0.0,6,True,False,0,True,True,True\n2026-07-29,FED-26JUL,2026-07-28,3.688775510204082,10.897963960864157,-4.49999999999999,4.49999999999999,117143.16,1.0,12.0,2026-07-28,71.7286298577407,2.194470731388488,0.7142857142856996,6,True,False,0,True,True,True\n2026-09-16,FED-26SEP,2026-09-15,3.846938775510204,7.891687907274463,5.49999999999997,5.49999999999997,213631.73,1.0,12.0,2026-09-15,74.37237444428611,5.037861647482482,1.7346938775510126,6,True,False,0,True,True,True\n', 'trades_csv': 'meeting_date,observation_date,event_key,threshold_pct,ticker,status,pages,query_start_utc,cutoff_end_utc,retrieved_utc,trades_retrieved,latest_timestamp_utc,age_hours,latest_batch_trades,latest_batch_distinct_prices,latest_batch_min_price,latest_batch_max_price,latest_batch_volume,latest_batch_vwap,full_48h_volume,full_48h_trades,api_endpoint\n2022-01-26,2022-01-25,FED-22JAN,0.25,FED-22JAN-T0.25,complete,1.0,2022-01-24T00:00:00+00:00,2022-01-26T00:00:00+00:00,2026-10-19T20:49:26.456794+00:00,16.0,2022-01-25T21:37:59.823339+00:00,2.3667157391666667,1.0,1.0,0.04,0.04,100.0,0.04,4151.0,16.0,/historical/trades\n2022-03-16,2022-03-15,FED-22MAR,0.25,FED-22MAR-T0.25,complete,1.0,2022-03-14T00:00:00+00:00,2022-03-16T00:00:00+00:00,2026-10-19T20:49:26.515407+00:00,32.0,2022-03-15T23:43:00.469023+00:00,0.2832030491666666,1.0,1.0,0.98,0.98,175.0,0.98,12588.0,32.0,/historical/trades\n2022-03-16,2022-03-15,FED-22MAR,0.5,FED-22MAR-T0.5,complete,1.0,2022-03-14T00:00:00+00:00,2022-03-16T00:00:00+00:00,2026-10-19T20:49:26.502416+00:00,42.0,2022-03-15T23:43:52.039192+00:00,0.2688780022222222,1.0,1.0,0.06,0.06,9.0,0.06,10842.0,42.0,/historical/trades\n2022-03-16,2022-03-15,FED-22MAR,0.75,FED-22MAR-T0.75,complete,1.0,2022-03-14T00:00:00+00:00,2022-03-16T00:00:00+00:00,2026-10-19T20:49:30.184939+00:00,13.0,2022-03-15T16:24:53.000448+00:00,7.585277653333334,1.0,1.0,0.02,0.02,93.0,0.02,2500.0,13.0,/historical/trades\n2022-05-04,2022-05-03,FED-22MAY,0.75,FED-22MAY-T0.75,complete,1.0,2022-05-02T00:00:00+00:00,2022-05-04T00:00:00+00:00,2026-10-19T20:49:30.241514+00:00,66.0,2022-05-03T21:18:15.258113+00:00,2.695761635277778,1.0,1.0,0.93,0.93,50.0,0.93,22497.0,66.0,/historical/trades\n2022-05-04,2022-05-03,FED-22MAY,1.0,FED-22MAY-T1,complete,1.0,2022-05-02T00:00:00+00:00,2022-05-04T00:00:00+00:00,2026-10-19T20:49:30.252588+00:00,8.0,2022-05-03T18:50:01.354497+00:00,5.1662904175,1.0,1.0,0.05,0.05,100.0,0.05,3844.0,8.0,/historical/trades\n2022-06-15,2022-06-14,FED-22JUN,0.25,FED-22JUN-T0.25,complete,1.0,2022-06-13T00:00:00+00:00,2022-06-15T00:00:00+00:00,2026-10-19T20:49:34.086678+00:00,4.0,2022-06-13T17:50:01.923728+00:00,30.16613229777778,3.0,1.0,0.99,0.99,934.0,0.99,936.0,4.0,/historical/trades\n2022-06-15,2022-06-14,FED-22JUN,0.5,FED-22JUN-T0.5,complete,1.0,2022-06-13T00:00:00+00:00,2022-06-15T00:00:00+00:00,2026-10-19T20:49:34.149307+00:00,8.0,2022-06-14T19:39:22.473390+00:00,4.343757391666667,2.0,1.0,0.99,0.99,100.0,0.99,5100.0,8.0,/historical/trades\n2022-06-15,2022-06-14,FED-22JUN,0.75,FED-22JUN-T0.75,complete,1.0,2022-06-13T00:00:00+00:00,2022-06-15T00:00:00+00:00,2026-10-19T20:49:34.093210+00:00,2.0,2022-06-13T20:41:13.359214+00:00,27.31295577388889,1.0,1.0,0.99,0.99,300.0,0.99,325.0,2.0,/historical/trades\n2022-06-15,2022-06-14,FED-22JUN,1.0,FED-22JUN-T1,complete,1.0,2022-06-13T00:00:00+00:00,2022-06-15T00:00:00+00:00,2026-10-19T20:49:37.984801+00:00,6.0,2022-06-14T19:05:48.963465+00:00,4.903065704166666,2.0,1.0,0.99,0.99,400.0,0.99,1076.0,6.0,/historical/trades\n2022-06-15,2022-06-14,FED-22JUN,1.25,FED-22JUN-T1.25,complete,1.0,2022-06-13T00:00:00+00:00,2022-06-15T00:00:00+00:00,2026-10-19T20:49:38.029510+00:00,31.0,2022-06-14T22:35:55.322592+00:00,1.40129928,1.0,1.0,0.97,0.97,100.0,0.97,8645.0,31.0,/historical/trades\n2022-06-15,2022-06-14,FED-22JUN,1.5,FED-22JUN-T1.50,complete,1.0,2022-06-13T00:00:00+00:00,2022-06-15T00:00:00+00:00,2026-10-19T20:49:38.076826+00:00,735.0,2022-06-14T23:49:03.024105+00:00,0.1824933041666666,1.0,1.0,0.8,0.8,100.0,0.8,138306.0,735.0,/historical/trades\n2022-06-15,2022-06-14,FED-22JUN,1.75,FED-22JUN-T1.75,complete,1.0,2022-06-13T00:00:00+00:00,2022-06-15T00:00:00+00:00,2026-10-19T20:49:41.845772+00:00,116.0,2022-06-14T23:47:22.243159+00:00,0.2104880113888889,1.0,1.0,0.06,0.06,25.0,0.06,43913.0,116.0,/historical/trades\n2022-07-27,2022-07-26,FED-22JUL,0.5,FED-22JUL-T0.5,complete,1.0,2022-07-25T00:00:00+00:00,2022-07-27T00:00:00+00:00,2026-10-19T20:49:41.694951+00:00,1.0,2022-07-26T13:51:31.638574+00:00,10.141211507222224,1.0,1.0,0.99,0.99,200.0,0.99,200.0,1.0,/historical/trades\n2022-07-27,2022-07-26,FED-22JUL,2.0,FED-22JUL-T2.00,complete,1.0,2022-07-25T00:00:00+00:00,2022-07-27T00:00:00+00:00,2026-10-19T20:49:41.705682+00:00,2.0,2022-07-26T20:44:38.309982+00:00,3.256025005,1.0,1.0,0.99,0.99,161.0,0.99,186.0,2.0,/historical/trades\n2022-07-27,2022-07-26,FED-22JUL,2.25,FED-22JULY-T2.25,complete,1.0,2022-07-25T00:00:00+00:00,2022-07-27T00:00:00+00:00,2026-10-19T20:49:45.626460+00:00,30.0,2022-07-26T23:45:17.307864+00:00,0.24519226,2.0,1.0,0.97,0.97,312.0,0.97,11198.0,30.0,/historical/trades\n2022-07-27,2022-07-26,FED-22JUL,2.5,FED-22JULY-T2.50,complete,1.0,2022-07-25T00:00:00+00:00,2022-07-27T00:00:00+00:00,2026-10-19T20:49:45.640468+00:00,166.0,2022-07-26T23:46:53.395921+00:00,0.2185011330555555,1.0,1.0,0.1,0.1,188.0,0.1,38780.0,166.0,/historical/trades\n2022-09-21,2022-09-20,FED-22SEP,2.75,FED-22SEP-T2.75,complete,1.0,2022-09-19T00:00:00+00:00,2022-09-21T00:00:00+00:00,2026-10-19T20:49:45.575988+00:00,3.0,2022-09-20T20:21:47.316546+00:00,3.636856515,1.0,1.0,0.99,0.99,100.0,0.99,1200.0,3.0,/historical/trades\n2022-09-21,2022-09-20,FED-22SEP,3.0,FED-22SEP-T3.00,complete,1.0,2022-09-19T00:00:00+00:00,2022-09-21T00:00:00+00:00,2026-10-19T20:50:03.299699+00:00,56.0,2022-09-20T23:15:21.210823+00:00,0.7441081047222222,6.0,2.0,0.97,0.98,3214.0,0.9781331673926572,40333.0,56.0,/historical/trades\n2022-09-21,2022-09-20,FED-22SEP,3.25,FED-22SEP-T3.25,complete,1.0,2022-09-19T00:00:00+00:00,2022-09-21T00:00:00+00:00,2026-10-19T20:50:03.327156+00:00,222.0,2022-09-20T23:41:04.227241+00:00,0.3154924330555555,1.0,1.0,0.07,0.07,446.0,0.07,56365.0,222.0,/historical/trades\n2022-09-21,2022-09-20,FED-22SEP,3.5,FED-22SEP-T3.50,complete,1.0,2022-09-19T00:00:00+00:00,2022-09-21T00:00:00+00:00,2026-10-19T20:50:03.349056+00:00,2.0,2022-09-20T16:16:27.741733+00:00,7.725627296388889,1.0,1.0,0.01,0.01,2525.0,0.01,3025.0,2.0,/historical/trades\n2022-11-02,2022-11-01,FED-22NOV,3.5,FED-22NOV-T3.50,complete,1.0,2022-10-31T00:00:00+00:00,2022-11-02T00:00:00+00:00,2026-10-19T20:50:08.013883+00:00,1.0,2022-11-01T18:02:36.411251+00:00,5.956552430277777,1.0,1.0,0.99,0.99,200.0,0.99,200.0,1.0,/historical/trades\n2022-11-02,2022-11-01,FED-22NOV,3.75,FED-22NOV-T3.75,complete,1.0,2022-10-31T00:00:00+00:00,2022-11-02T00:00:00+00:00,2026-10-19T20:50:07.772542+00:00,83.0,2022-11-01T23:49:58.706187+00:00,0.1670260591666666,1.0,1.0,0.98,0.98,26.0,0.98,38242.0,83.0,/historical/trades\n2022-11-02,2022-11-01,FED-22NOV,4.0,FED-22NOV-T4.00,complete,1.0,2022-10-31T00:00:00+00:00,2022-11-02T00:00:00+00:00,2026-10-19T20:50:08.008560+00:00,12.0,2022-11-01T23:25:05.211686+00:00,0.5818856427777778,1.0,1.0,0.02,0.02,50.0,0.02,1473.0,12.0,/historical/trades\n2022-12-14,2022-12-13,FED-22DEC,0.75,FED-22DEC-T0.75,complete,1.0,2022-12-12T00:00:00+00:00,2022-12-14T00:00:00+00:00,2026-10-19T20:50:11.832505+00:00,2.0,2022-12-13T13:40:10.850167+00:00,10.330319398055556,2.0,1.0,0.99,0.99,500.0,0.99,500.0,2.0,/historical/trades\n2022-12-14,2022-12-13,FED-22DEC,4.0,FED-22DEC-T4.00,complete,1.0,2022-12-12T00:00:00+00:00,2022-12-14T00:00:00+00:00,2026-10-19T20:50:11.836435+00:00,1.0,2022-12-12T20:57:45.642160+00:00,27.037321622222223,1.0,1.0,0.99,0.99,108.0,0.99,108.0,1.0,/historical/trades\n2022-12-14,2022-12-13,FED-22DEC,4.25,FED-22DEC-T4.25,complete,1.0,2022-12-12T00:00:00+00:00,2022-12-14T00:00:00+00:00,2026-10-19T20:50:11.879790+00:00,77.0,2022-12-13T23:15:48.568930+00:00,0.7365086305555556,1.0,1.0,0.99,0.99,10.0,0.99,26586.0,77.0,/historical/trades\n2022-12-14,2022-12-13,FED-22DEC,4.5,FED-22DEC-T4.50,complete,1.0,2022-12-12T00:00:00+00:00,2022-12-14T00:00:00+00:00,2026-10-19T20:50:15.709808+00:00,76.0,2022-12-13T23:16:38.903664+00:00,0.72252676,1.0,1.0,0.01,0.01,100.0,0.01,65763.0,76.0,/historical/trades\n2022-12-14,2022-12-13,FED-22DEC,4.75,FED-22DEC-T4.75,complete,1.0,2022-12-12T00:00:00+00:00,2022-12-14T00:00:00+00:00,2026-10-19T20:50:15.683816+00:00,1.0,2022-12-12T19:48:56.341302+00:00,28.18434963833333,1.0,1.0,0.01,0.01,330.0,0.01,330.0,1.0,/historical/trades\n2023-02-01,2023-01-31,FED-23FEB,3.25,FED-23FEB-T3.25,complete,1.0,2023-01-30T00:00:00+00:00,2023-02-01T00:00:00+00:00,2026-10-19T20:46:51.251909+00:00,1.0,2023-01-30T19:11:00.581252+00:00,28.816505207777777,1.0,1.0,0.99,0.99,5.0,0.99,5.0,1.0,/historical/trades\n2023-02-01,2023-01-31,FED-23FEB,4.0,FED-23FEB-T4.00,complete,1.0,2023-01-30T00:00:00+00:00,2023-02-01T00:00:00+00:00,2026-10-19T20:46:51.312594+00:00,1.0,2023-01-31T01:44:34.384993+00:00,22.25711527972222,1.0,1.0,0.99,0.99,27.0,0.99,27.0,1.0,/historical/trades\n2023-02-01,2023-01-31,FED-23FEB,4.5,FED-23FEB-T4.50,complete,1.0,2023-01-30T00:00:00+00:00,2023-02-01T00:00:00+00:00,2026-10-19T20:46:51.244123+00:00,23.0,2023-01-31T15:23:16.266850+00:00,8.612148097222223,2.0,1.0,0.98,0.98,415.0,0.98,5721.0,23.0,/historical/trades\n2023-02-01,2023-01-31,FED-23FEB,4.75,FED-23FEB-T4.75,complete,1.0,2023-01-30T00:00:00+00:00,2023-02-01T00:00:00+00:00,2026-10-19T20:46:54.529645+00:00,42.0,2023-01-31T21:33:20.957089+00:00,2.444178586388889,1.0,1.0,0.02,0.02,50.0,0.02,10108.0,42.0,/historical/trades\n2023-03-22,2023-03-21,FED-23MAR,3.0,FED-23MAR-T3.00,complete,1.0,2023-03-20T00:00:00+00:00,2023-03-22T00:00:00+00:00,2026-10-19T20:46:54.486002+00:00,1.0,2023-03-21T00:13:02.294569+00:00,23.782695953055555,1.0,1.0,0.99,0.99,100.0,0.99,100.0,1.0,/historical/trades\n2023-03-22,2023-03-21,FED-23MAR,4.25,FED-23MAR-T4.25,complete,1.0,2023-03-20T00:00:00+00:00,2023-03-22T00:00:00+00:00,2026-10-19T20:46:54.494006+00:00,1.0,2023-03-21T18:17:32.027801+00:00,5.707770055277778,1.0,1.0,0.99,0.99,10.0,0.99,10.0,1.0,/historical/trades\n2023-03-22,2023-03-21,FED-23MAR,4.5,FED-23MAR-T4.50,complete,1.0,2023-03-20T00:00:00+00:00,2023-03-22T00:00:00+00:00,2026-10-19T20:46:59.213454+00:00,21.0,2023-03-21T20:27:33.930287+00:00,3.540574920277778,1.0,1.0,0.99,0.99,193.0,0.99,3880.0,21.0,/historical/trades\n2023-03-22,2023-03-21,FED-23MAR,4.75,FED-23MAR-T4.75,complete,1.0,2023-03-20T00:00:00+00:00,2023-03-22T00:00:00+00:00,2026-10-19T20:46:59.441517+00:00,302.0,2023-03-21T23:59:42.516195+00:00,0.0048566125,2.0,1.0,0.82,0.82,169.0,0.82,64412.0,302.0,/historical/trades\n2023-03-22,2023-03-21,FED-23MAR,5.0,FED-23MAR-T5.00,complete,1.0,2023-03-20T00:00:00+00:00,2023-03-22T00:00:00+00:00,2026-10-19T20:46:59.222733+00:00,94.0,2023-03-21T23:45:20.912803+00:00,0.2441908880555555,5.0,3.0,0.03,0.05,11746.0,0.0381151030137919,82654.0,94.0,/historical/trades\n2023-03-22,2023-03-21,FED-23MAR,5.25,FED-23MAR-T5.25,complete,1.0,2023-03-20T00:00:00+00:00,2023-03-22T00:00:00+00:00,2026-10-19T20:47:03.437168+00:00,3.0,2023-03-21T20:52:38.221625+00:00,3.122716215277778,3.0,1.0,0.01,0.01,600.0,0.01,600.0,3.0,/historical/trades\n2023-05-03,2023-05-02,FED-23MAY,4.75,FED-23MAY-T4.75,complete,1.0,2023-05-01T00:00:00+00:00,2023-05-03T00:00:00+00:00,2026-10-19T20:47:03.554402+00:00,12.0,2023-05-02T19:25:57.751420+00:00,4.567291272222222,1.0,1.0,0.99,0.99,75.0,0.99,9721.0,12.0,/historical/trades\n2023-05-03,2023-05-02,FED-23MAY,5.0,FED-23MAY-T5.00,complete,1.0,2023-05-01T00:00:00+00:00,2023-05-03T00:00:00+00:00,2026-10-19T20:47:03.573810+00:00,252.0,2023-05-02T23:49:37.842837+00:00,0.1728214341666666,2.0,1.0,0.93,0.93,75.0,0.93,108229.0,252.0,/historical/trades\n2023-05-03,2023-05-02,FED-23MAY,5.25,FED-23MAY-T5.25,complete,1.0,2023-05-01T00:00:00+00:00,2023-05-03T00:00:00+00:00,2026-10-19T20:47:07.431605+00:00,13.0,2023-05-02T19:25:33.997967+00:00,4.573889453611112,1.0,1.0,0.01,0.01,10.0,0.01,12325.0,13.0,/historical/trades\n2023-06-14,2023-06-13,FED-23JUN,5.0,FED-23JUN-T5.00,complete,1.0,2023-06-12T00:00:00+00:00,2023-06-14T00:00:00+00:00,2026-10-19T20:47:07.439854+00:00,1.0,2023-06-13T01:45:53.378779+00:00,22.23517256138889,1.0,1.0,0.99,0.99,280.0,0.99,280.0,1.0,/historical/trades\n2023-06-14,2023-06-13,FED-23JUN,5.25,FED-23JUN-T5.25,complete,1.0,2023-06-12T00:00:00+00:00,2023-06-14T00:00:00+00:00,2026-10-19T20:47:07.492922+00:00,278.0,2023-06-13T23:52:40.609182+00:00,0.122053005,1.0,1.0,0.05,0.05,2416.0,0.05,97052.0,278.0,/historical/trades\n2023-06-14,2023-06-13,FED-23JUN,5.5,FED-23JUN-T5.50,complete,1.0,2023-06-12T00:00:00+00:00,2023-06-14T00:00:00+00:00,2026-10-19T20:47:12.222320+00:00,3.0,2023-06-13T13:27:55.636633+00:00,10.53454537972222,1.0,1.0,0.01,0.01,13527.0,0.01,23527.0,3.0,/historical/trades\n2023-07-26,2023-07-25,FED-23JUL,5.25,FED-23JUL-T5.25,complete,1.0,2023-07-24T00:00:00+00:00,2023-07-26T00:00:00+00:00,2026-10-19T20:47:12.299349+00:00,14.0,2023-07-25T16:52:46.198368+00:00,7.120500453333333,1.0,1.0,0.99,0.99,1000.0,0.99,12655.0,14.0,/historical/trades\n2023-07-26,2023-07-25,FED-23JUL,5.5,FED-23JUL-T5.50,complete,1.0,2023-07-24T00:00:00+00:00,2023-07-26T00:00:00+00:00,2026-10-19T20:47:12.276301+00:00,1.0,2023-07-24T17:51:37.089543+00:00,30.139697349166667,1.0,1.0,0.01,0.01,1000.0,0.01,1000.0,1.0,/historical/trades\n2023-09-20,2023-09-19,FED-23SEP,5.25,FED-23SEP-T5.25,complete,1.0,2023-09-18T00:00:00+00:00,2023-09-20T00:00:00+00:00,2026-10-19T20:47:16.217491+00:00,5.0,2023-09-18T13:35:47.050072+00:00,34.40359720222222,1.0,1.0,0.99,0.99,156.0,0.99,215.0,5.0,/historical/trades\n2023-09-20,2023-09-19,FED-23SEP,5.5,FED-23SEP-T5.50,complete,1.0,2023-09-18T00:00:00+00:00,2023-09-20T00:00:00+00:00,2026-10-19T20:47:16.325827+00:00,47.0,2023-09-19T22:58:02.485595+00:00,1.0326428902777778,1.0,1.0,0.02,0.02,469.0,0.02,16725.0,47.0,/historical/trades\n2023-09-20,2023-09-19,FED-23SEP,5.75,FED-23SEP-T5.75,complete,1.0,2023-09-18T00:00:00+00:00,2023-09-20T00:00:00+00:00,2026-10-19T20:47:16.210834+00:00,5.0,2023-09-18T17:27:52.667729+00:00,30.53537007527778,1.0,1.0,0.01,0.01,4937.0,0.01,5787.0,5.0,/historical/trades\n2023-09-20,2023-09-19,FED-23SEP,6.5,FED-23SEP-T6.50,complete,1.0,2023-09-18T00:00:00+00:00,2023-09-20T00:00:00+00:00,2026-10-19T20:47:19.569687+00:00,2.0,2023-09-18T13:36:24.020295+00:00,34.393327695833335,1.0,1.0,0.01,0.01,3600.0,0.01,3700.0,2.0,/historical/trades\n2023-11-01,2023-10-31,FED-23NOV,4.25,FED-23NOV-T4.25,complete,1.0,2023-10-30T00:00:00+00:00,2023-11-01T00:00:00+00:00,2026-10-19T20:47:19.588022+00:00,1.0,2023-10-30T19:43:42.942760+00:00,28.271404788888887,1.0,1.0,0.99,0.99,10.0,0.99,10.0,1.0,/historical/trades\n2023-11-01,2023-10-31,FED-23NOV,5.25,FED-23NOV-T5.25,complete,1.0,2023-10-30T00:00:00+00:00,2023-11-01T00:00:00+00:00,2026-10-19T20:47:19.523624+00:00,2.0,2023-10-31T02:42:05.051329+00:00,21.29859685305556,1.0,1.0,0.99,0.99,93.0,0.99,143.0,2.0,/historical/trades\n2023-11-01,2023-10-31,FED-23NOV,5.5,FED-23NOV-T5.50,complete,1.0,2023-10-30T00:00:00+00:00,2023-11-01T00:00:00+00:00,2026-10-19T20:47:26.560391+00:00,24.0,2023-10-31T21:42:15.213569+00:00,2.295774008611111,1.0,1.0,0.01,0.01,1000.0,0.01,16619.0,24.0,/historical/trades\n2024-01-31,2024-01-30,FED-24JAN,5.5,FED-24JAN-T5.50,complete,1.0,2024-01-29T00:00:00+00:00,2024-01-31T00:00:00+00:00,2026-10-19T20:47:26.513610+00:00,3.0,2024-01-30T14:32:23.611694+00:00,9.46010786277778,3.0,1.0,0.01,0.01,2000.0,0.01,2000.0,3.0,/historical/trades\n2024-03-20,2024-03-19,FED-24MAR,5.25,FED-24MAR-T5.25,complete,1.0,2024-03-18T00:00:00+00:00,2024-03-20T00:00:00+00:00,2026-10-19T20:47:26.519693+00:00,4.0,2024-03-19T20:41:06.263230+00:00,3.3149268805555554,1.0,1.0,0.99,0.99,2000.0,0.99,11327.0,4.0,/historical/trades\n2024-03-20,2024-03-19,FED-24MAR,5.5,FED-24MAR-T5.50,complete,1.0,2024-03-18T00:00:00+00:00,2024-03-20T00:00:00+00:00,2026-10-19T20:47:31.317021+00:00,2.0,2024-03-18T02:33:01.011128+00:00,45.44971913111111,2.0,1.0,0.01,0.01,93.0,0.01,93.0,2.0,/historical/trades\n2024-05-01,2024-04-30,FED-24MAY,5.25,FED-24MAY-T5.25,complete,1.0,2024-04-29T00:00:00+00:00,2024-05-01T00:00:00+00:00,2026-10-19T20:47:31.305823+00:00,1.0,2024-04-30T16:48:52.718987+00:00,7.185355836944445,1.0,1.0,0.99,0.99,200.0,0.99,200.0,1.0,/historical/trades\n2024-05-01,2024-04-30,FED-24MAY,5.5,FED-24MAY-T5.50,complete,1.0,2024-04-29T00:00:00+00:00,2024-05-01T00:00:00+00:00,2026-10-19T20:47:31.311848+00:00,1.0,2024-04-30T18:41:25.122371+00:00,5.309688230277778,1.0,1.0,0.01,0.01,150.0,0.01,150.0,1.0,/historical/trades\n2024-06-12,2024-06-11,FED-24JUN,5.25,FED-24JUN-T5.25,complete,1.0,2024-06-10T00:00:00+00:00,2024-06-12T00:00:00+00:00,2026-10-19T20:47:35.116274+00:00,31.0,2024-06-11T20:50:46.218718+00:00,3.1538281338888887,2.0,1.0,0.99,0.99,5185.0,0.99,32357.0,31.0,/historical/trades\n2024-06-12,2024-06-11,FED-24JUN,5.5,FED-24JUN-T5.50,complete,1.0,2024-06-10T00:00:00+00:00,2024-06-12T00:00:00+00:00,2026-10-19T20:47:35.143777+00:00,2.0,2024-06-11T15:56:53.309967+00:00,8.0518583425,1.0,1.0,0.01,0.01,1125.0,0.01,1592.0,2.0,/historical/trades\n2024-07-31,2024-07-30,FED-24JUL,5.25,FED-24JUL-T5.25,complete,1.0,2024-07-29T00:00:00+00:00,2024-07-31T00:00:00+00:00,2026-10-19T20:47:35.138551+00:00,6.0,2024-07-30T14:11:10.185578+00:00,9.813837339444444,1.0,1.0,0.98,0.98,1.0,0.98,1629.0,6.0,/historical/trades\n2024-09-18,2024-09-17,FED-24SEP,4.25,FED-24SEP-T4.25,complete,1.0,2024-09-16T00:00:00+00:00,2024-09-18T00:00:00+00:00,2026-10-19T20:47:38.291998+00:00,2.0,2024-09-16T17:26:24.796687+00:00,30.55977869805556,2.0,1.0,0.99,0.99,353.0,0.99,353.0,2.0,/historical/trades\n2024-09-18,2024-09-17,FED-24SEP,4.5,FED-24SEP-T4.50,complete,1.0,2024-09-16T00:00:00+00:00,2024-09-18T00:00:00+00:00,2026-10-19T20:47:38.298772+00:00,1.0,2024-09-16T17:26:39.957456+00:00,30.55556737333333,1.0,1.0,0.99,0.99,118.0,0.99,118.0,1.0,/historical/trades\n2024-09-18,2024-09-17,FED-24SEP,4.75,FED-24SEP-T4.75,complete,1.0,2024-09-16T00:00:00+00:00,2024-09-18T00:00:00+00:00,2026-10-19T20:47:38.304824+00:00,17.0,2024-09-17T23:50:17.425255+00:00,0.1618263180555555,1.0,1.0,0.99,0.99,145.0,0.99,4227.0,17.0,/historical/trades\n2024-09-18,2024-09-17,FED-24SEP,5.0,FED-24SEP-T5.00,complete,1.0,2024-09-16T00:00:00+00:00,2024-09-18T00:00:00+00:00,2026-10-19T20:47:41.534481+00:00,73.0,2024-09-17T22:29:41.852603+00:00,1.505040943611111,1.0,1.0,0.47,0.47,205.0,0.47,30772.0,73.0,/historical/trades\n2024-09-18,2024-09-17,FED-24SEP,5.25,FED-24SEP-T5.25,complete,1.0,2024-09-16T00:00:00+00:00,2024-09-18T00:00:00+00:00,2026-10-19T20:47:41.556180+00:00,4.0,2024-09-17T21:33:08.667322+00:00,2.4475924105555555,1.0,1.0,0.01,0.01,13.0,0.01,3327.0,4.0,/historical/trades\n2024-11-07,2024-11-06,FED-24NOV,4.5,FED-24NOV-T4.50,complete,1.0,2024-11-05T00:00:00+00:00,2024-11-07T00:00:00+00:00,2026-10-19T20:47:41.489272+00:00,2.0,2024-11-06T02:15:32.849985+00:00,21.740875004166668,1.0,1.0,0.99,0.99,208.0,0.99,520.0,2.0,/historical/trades\n2024-11-07,2024-11-06,FED-24NOV,4.75,FED-24NOV-T4.75,complete,1.0,2024-11-05T00:00:00+00:00,2024-11-07T00:00:00+00:00,2026-10-19T20:47:45.070228+00:00,7.0,2024-11-06T18:34:58.293625+00:00,5.417140659722223,2.0,2.0,0.02,0.03,21000.0,0.0200023809523809,24049.0,7.0,/historical/trades\n2024-12-18,2024-12-17,FED-24DEC,4.25,FED-24DEC-T4.25,complete,1.0,2024-12-16T00:00:00+00:00,2024-12-18T00:00:00+00:00,2026-10-19T20:47:45.181970+00:00,4.0,2024-12-17T04:10:12.513454+00:00,19.82985737388889,1.0,1.0,0.98,0.98,163.0,0.98,2506.0,4.0,/historical/trades\n2024-12-18,2024-12-17,FED-24DEC,4.5,FED-24DEC-T4.50,complete,1.0,2024-12-16T00:00:00+00:00,2024-12-18T00:00:00+00:00,2026-10-19T20:47:45.171436+00:00,31.0,2024-12-17T23:42:38.027877+00:00,0.2894367008333333,1.0,1.0,0.04,0.04,35.0,0.04,11700.0,31.0,/historical/trades\n2024-12-18,2024-12-17,FED-24DEC,4.75,FED-24DEC-T4.75,complete,1.0,2024-12-16T00:00:00+00:00,2024-12-18T00:00:00+00:00,2026-10-19T20:47:51.963067+00:00,2.0,2024-12-16T19:50:00.996106+00:00,28.166389970555556,1.0,1.0,0.03,0.03,312.0,0.03,468.0,2.0,/historical/trades\n2025-01-29,2025-01-28,FED-25JAN,4.0,FED-25JAN-T4.00,complete,1.0,2025-01-27T00:00:00+00:00,2025-01-29T00:00:00+00:00,2026-10-19T20:50:15.670730+00:00,1.0,2025-01-27T14:27:34.220756+00:00,33.540494234444445,1.0,1.0,0.99,0.99,4000.0,0.99,4000.0,1.0,/historical/trades\n2025-01-29,2025-01-28,FED-25JAN,4.25,FED-25JAN-T4.25,complete,1.0,2025-01-27T00:00:00+00:00,2025-01-29T00:00:00+00:00,2026-10-19T20:50:19.684207+00:00,7.0,2025-01-28T15:48:13.046138+00:00,8.196376072777777,1.0,1.0,0.99,0.99,1041.0,0.99,9555.0,7.0,/historical/trades\n2025-01-29,2025-01-28,FED-25JAN,4.5,FED-25JAN-T4.50,complete,1.0,2025-01-27T00:00:00+00:00,2025-01-29T00:00:00+00:00,2026-10-19T20:50:19.675133+00:00,1.0,2025-01-27T16:00:03.123701+00:00,31.999132305277776,1.0,1.0,0.01,0.01,93.0,0.01,93.0,1.0,/historical/trades\n2025-03-19,2025-03-18,FED-25MAR,4.25,FED-25MAR-T4.25,complete,1.0,2025-03-17T00:00:00+00:00,2025-03-19T00:00:00+00:00,2026-10-19T20:47:51.937238+00:00,19.0,2025-03-18T21:57:40.491323+00:00,2.0387524102777776,1.0,1.0,0.99,0.99,1.0,0.99,1515.0,19.0,/historical/trades\n2025-03-19,2025-03-18,FED-25MAR,4.5,FED-25MAR-T4.50,complete,1.0,2025-03-17T00:00:00+00:00,2025-03-19T00:00:00+00:00,2026-10-19T20:47:51.897111+00:00,3.0,2025-03-17T22:59:52.579593+00:00,25.002061224166667,1.0,1.0,0.02,0.02,30.0,0.02,60.0,3.0,/historical/trades\n2025-05-07,2025-05-06,FED-25MAY,4.0,FED-25MAY-T4.00,complete,1.0,2025-05-05T00:00:00+00:00,2025-05-07T00:00:00+00:00,2026-10-19T20:47:57.509069+00:00,143.0,2025-05-06T23:48:33.888560+00:00,0.1905865111111111,1.0,1.0,0.99,0.99,5.0,0.99,73087.0,143.0,/historical/trades\n2025-05-07,2025-05-06,FED-25MAY,4.25,FED-25MAY-T4.25,complete,1.0,2025-05-05T00:00:00+00:00,2025-05-07T00:00:00+00:00,2026-10-19T20:47:57.500235+00:00,243.0,2025-05-06T23:57:08.567871+00:00,0.0476200358333333,1.0,1.0,0.97,0.97,10.0,0.97,127915.0,243.0,/historical/trades\n2025-05-07,2025-05-06,FED-25MAY,4.5,FED-25MAY-T4.50,complete,1.0,2025-05-05T00:00:00+00:00,2025-05-07T00:00:00+00:00,2026-10-19T20:47:57.447016+00:00,14.0,2025-05-06T21:51:32.597784+00:00,2.14094506,1.0,1.0,0.01,0.01,1.0,0.01,182.0,14.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,2.75,FED-25JUN-T2.75,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:02.830966+00:00,13.0,2025-06-17T22:37:24.367895+00:00,1.3765644736111111,1.0,1.0,0.99,0.99,100.0,0.99,30832.0,13.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,3.0,FED-25JUN-T3.00,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:02.782109+00:00,1.0,2025-06-17T21:36:00.020604+00:00,2.399994276666667,1.0,1.0,0.99,0.99,50.0,0.99,50.0,1.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,3.25,FED-25JUN-T3.25,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:02.772205+00:00,5.0,2025-06-17T21:35:46.197897+00:00,2.4038339175,1.0,1.0,0.99,0.99,50.0,0.99,10210.0,5.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,3.5,FED-25JUN-T3.50,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:06.872808+00:00,2.0,2025-06-17T21:35:34.563146+00:00,2.407065792777778,1.0,1.0,0.99,0.99,50.0,0.99,10050.0,2.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,3.75,FED-25JUN-T3.75,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:06.916117+00:00,3.0,2025-06-17T21:35:22.595131+00:00,2.410390241388889,1.0,1.0,0.99,0.99,50.0,0.99,12050.0,3.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,4.0,FED-25JUN-T4.00,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:06.941580+00:00,5.0,2025-06-17T21:34:59.988758+00:00,2.4166697894444447,1.0,1.0,0.99,0.99,10.0,0.99,2303.0,5.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,4.25,FED-25JUN-T4.25,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:10.959148+00:00,40.0,2025-06-17T22:03:26.312317+00:00,1.942691023055556,1.0,1.0,0.98,0.98,200.0,0.98,42337.0,40.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,4.5,FED-25JUN-T4.50,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:10.893365+00:00,57.0,2025-06-17T23:03:58.148604+00:00,0.93384761,1.0,1.0,0.01,0.01,500.0,0.01,9534.0,57.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,4.75,FED-25JUN-T4.75,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:10.848624+00:00,10.0,2025-06-17T23:40:38.531312+00:00,0.3226301911111111,1.0,1.0,0.01,0.01,10.0,0.01,1630.0,10.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,5.0,FED-25JUN-T5.00,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:16.938556+00:00,5.0,2025-06-17T23:40:10.852696+00:00,0.3303186955555556,1.0,1.0,0.01,0.01,10.0,0.01,230.0,5.0,/historical/trades\n2025-06-18,2025-06-17,FED-25JUN,5.25,FED-25JUN-T5.25,complete,1.0,2025-06-16T00:00:00+00:00,2025-06-18T00:00:00+00:00,2026-10-19T20:48:16.941702+00:00,4.0,2025-06-17T23:39:55.197049+00:00,0.3346674863888889,1.0,1.0,0.01,0.01,10.0,0.01,210.0,4.0,/historical/trades\n2025-07-30,2025-07-29,FED-25JUL,2.75,FED-25JUL-T2.75,complete,1.0,2025-07-28T00:00:00+00:00,2025-07-30T00:00:00+00:00,2026-10-19T20:48:17.000465+00:00,13.0,2025-07-29T23:54:44.123862+00:00,0.0877433716666666,1.0,1.0,0.99,0.99,1.0,0.99,11488.0,13.0,/historical/trades\n2025-07-30,2025-07-29,FED-25JUL,3.0,FED-25JUL-T3.00,complete,1.0,2025-07-28T00:00:00+00:00,2025-07-30T00:00:00+00:00,2026-10-19T20:48:20.712333+00:00,6.0,2025-07-29T20:27:34.160579+00:00,3.540510950277778,1.0,1.0,0.99,0.99,3.0,0.99,550.0,6.0,/historical/trades\n2025-07-30,2025-07-29,FED-25JUL,3.25,FED-25JUL-T3.25,complete,1.0,2025-07-28T00:00:00+00:00,2025-07-30T00:00:00+00:00,2026-10-19T20:48:20.643922+00:00,3.0,2025-07-29T14:17:23.747447+00:00,9.710070153611111,1.0,1.0,0.99,0.99,1.0,0.99,52.0,3.0,/historical/trades\n2025-07-30,2025-07-29,FED-25JUL,3.5,FED-25JUL-T3.50,complete,1.0,2025-07-28T00:00:00+00:00,2025-07-30T00:00:00+00:00,2026-10-19T20:48:20.648109+00:00,1.0,2025-07-28T18:27:54.566473+00:00,29.53484264638889,1.0,1.0,0.99,0.99,1.0,0.99,1.0,1.0,/historical/trades\n2025-07-30,2025-07-29,FED-25JUL,4.0,FED-25JUL-T4.00,complete,1.0,2025-07-28T00:00:00+00:00,2025-07-30T00:00:00+00:00,2026-10-19T20:48:24.551506+00:00,18.0,2025-07-29T16:27:28.557013+00:00,7.542067496388888,1.0,1.0,0.99,0.99,9351.0,0.99,43379.0,18.0,/historical/trades\n2025-07-30,2025-07-29,FED-25JUL,4.25,FED-25JUL-T4.25,complete,1.0,2025-07-28T00:00:00+00:00,2025-07-30T00:00:00+00:00,2026-10-19T20:48:24.666270+00:00,55.0,2025-07-29T23:03:19.715370+00:00,0.9445235083333332,1.0,1.0,0.97,0.97,3121.0,0.97,191571.0,55.0,/historical/trades\n2025-07-30,2025-07-29,FED-25JUL,4.5,FED-25JUL-T4.50,complete,1.0,2025-07-28T00:00:00+00:00,2025-07-30T00:00:00+00:00,2026-10-19T20:48:24.594240+00:00,28.0,2025-07-29T22:12:08.049327+00:00,1.7977640758333333,1.0,1.0,0.01,0.01,7.0,0.01,4626.0,28.0,/historical/trades\n2025-07-30,2025-07-29,FED-25JUL,4.75,FED-25JUL-T4.75,complete,1.0,2025-07-28T00:00:00+00:00,2025-07-30T00:00:00+00:00,2026-10-19T20:48:28.389131+00:00,1.0,2025-07-29T16:15:28.294120+00:00,7.742140522222223,1.0,1.0,0.01,0.01,93.0,0.01,93.0,1.0,/historical/trades\n2025-07-30,2025-07-29,FED-25JUL,5.0,FED-25JUL-T5.00,complete,1.0,2025-07-28T00:00:00+00:00,2025-07-30T00:00:00+00:00,2026-10-19T20:48:28.439636+00:00,1.0,2025-07-29T16:15:54.674152+00:00,7.734812735555556,1.0,1.0,0.01,0.01,93.0,0.01,93.0,1.0,/historical/trades\n2025-07-30,2025-07-29,FED-25JUL,5.25,FED-25JUL-T5.25,complete,1.0,2025-07-28T00:00:00+00:00,2025-07-30T00:00:00+00:00,2026-10-19T20:48:28.454598+00:00,2.0,2025-07-28T23:02:48.731321+00:00,24.953130188611112,1.0,1.0,0.01,0.01,14.0,0.01,114.0,2.0,/historical/trades\n2025-09-17,2025-09-16,FED-25SEP,2.75,FED-25SEP-T2.75,complete,1.0,2025-09-15T00:00:00+00:00,2025-09-17T00:00:00+00:00,2026-10-19T20:48:32.250163+00:00,5.0,2025-09-16T02:25:45.555056+00:00,21.57067915111111,1.0,1.0,0.99,0.99,5.0,0.99,169.0,5.0,/historical/trades\n2025-09-17,2025-09-16,FED-25SEP,3.0,FED-25SEP-T3.00,complete,1.0,2025-09-15T00:00:00+00:00,2025-09-17T00:00:00+00:00,2026-10-19T20:48:32.315863+00:00,3.0,2025-09-16T22:08:06.450087+00:00,1.8648749758333332,1.0,1.0,0.99,0.99,6.0,0.99,66.0,3.0,/historical/trades\n2025-09-17,2025-09-16,FED-25SEP,3.25,FED-25SEP-T3.25,complete,1.0,2025-09-15T00:00:00+00:00,2025-09-17T00:00:00+00:00,2026-10-19T20:48:32.296142+00:00,1.0,2025-09-15T03:26:49.904581+00:00,44.55280428305555,1.0,1.0,0.99,0.99,50.0,0.99,50.0,1.0,/historical/trades\n2025-09-17,2025-09-16,FED-25SEP,3.5,FED-25SEP-T3.50,complete,1.0,2025-09-15T00:00:00+00:00,2025-09-17T00:00:00+00:00,2026-10-19T20:48:36.108327+00:00,4.0,2025-09-15T14:34:22.995936+00:00,33.426945573333334,1.0,1.0,0.99,0.99,30.0,0.99,133.0,4.0,/historical/trades\n2025-09-17,2025-09-16,FED-25SEP,3.75,FED-25SEP-T3.75,complete,1.0,2025-09-15T00:00:00+00:00,2025-09-17T00:00:00+00:00,2026-10-19T20:48:36.177464+00:00,25.0,2025-09-16T22:17:08.957684+00:00,1.714178421111111,1.0,1.0,0.99,0.99,93.0,0.99,6366.0,25.0,/historical/trades\n2025-09-17,2025-09-16,FED-25SEP,4.0,FED-25SEP-T4.00,complete,1.0,2025-09-15T00:00:00+00:00,2025-09-17T00:00:00+00:00,2026-10-19T20:48:36.157778+00:00,80.0,2025-09-16T23:30:25.629181+00:00,0.4928807830555555,1.0,1.0,0.94,0.94,200.0,0.94,94482.0,80.0,/historical/trades\n2025-09-17,2025-09-16,FED-25SEP,4.25,FED-25SEP-T4.25,complete,1.0,2025-09-15T00:00:00+00:00,2025-09-17T00:00:00+00:00,2026-10-19T20:48:39.859804+00:00,105.0,2025-09-16T23:19:19.785314+00:00,0.6778374127777778,1.0,1.0,0.03,0.03,156.0,0.03,90646.0,105.0,/historical/trades\n2025-09-17,2025-09-16,FED-25SEP,4.5,FED-25SEP-T4.50,complete,1.0,2025-09-15T00:00:00+00:00,2025-09-17T00:00:00+00:00,2026-10-19T20:48:39.813376+00:00,18.0,2025-09-16T20:26:23.994671+00:00,3.5600014802777777,1.0,1.0,0.01,0.01,10.0,0.01,1141.0,18.0,/historical/trades\n2025-09-17,2025-09-16,FED-25SEP,4.75,FED-25SEP-T4.75,complete,1.0,2025-09-15T00:00:00+00:00,2025-09-17T00:00:00+00:00,2026-10-19T20:48:39.879422+00:00,1.0,2025-09-15T14:36:40.175401+00:00,33.38884016638889,1.0,1.0,0.01,0.01,98.0,0.01,98.0,1.0,/historical/trades\n2025-09-17,2025-09-16,FED-25SEP,5.0,FED-25SEP-T5.00,complete,1.0,2025-09-15T00:00:00+00:00,2025-09-17T00:00:00+00:00,2026-10-19T20:48:43.365269+00:00,1.0,2025-09-15T14:37:37.312746+00:00,33.37296868166667,1.0,1.0,0.01,0.01,18.0,0.01,18.0,1.0,/historical/trades\n2025-10-29,2025-10-28,FED-25OCT,2.75,FED-25OCT-T2.75,complete,1.0,2025-10-27T00:00:00+00:00,2025-10-29T00:00:00+00:00,2026-10-19T20:48:43.302689+00:00,6.0,2025-10-28T17:25:37.097765+00:00,6.573028398611111,1.0,1.0,0.99,0.99,100.0,0.99,116.0,6.0,/historical/trades\n2025-10-29,2025-10-28,FED-25OCT,3.5,FED-25OCT-T3.50,complete,1.0,2025-10-27T00:00:00+00:00,2025-10-29T00:00:00+00:00,2026-10-19T20:48:43.298084+00:00,8.0,2025-10-28T14:29:02.068654+00:00,9.516092040555554,2.0,1.0,0.99,0.99,93.0,0.99,1241.0,8.0,/historical/trades\n2025-10-29,2025-10-28,FED-25OCT,3.75,FED-25OCT-T3.75,complete,1.0,2025-10-27T00:00:00+00:00,2025-10-29T00:00:00+00:00,2026-10-19T20:48:47.134391+00:00,59.0,2025-10-28T23:33:03.545048+00:00,0.4490152644444444,1.0,1.0,0.98,0.98,172.0,0.98,15812.0,59.0,/historical/trades\n2025-10-29,2025-10-28,FED-25OCT,4.0,FED-25OCT-T4.00,complete,1.0,2025-10-27T00:00:00+00:00,2025-10-29T00:00:00+00:00,2026-10-19T20:48:47.036873+00:00,57.0,2025-10-28T22:55:24.693700+00:00,1.0764739722222223,1.0,1.0,0.02,0.02,4679.0,0.02,50715.0,57.0,/historical/trades\n2025-10-29,2025-10-28,FED-25OCT,4.25,FED-25OCT-T4.25,complete,1.0,2025-10-27T00:00:00+00:00,2025-10-29T00:00:00+00:00,2026-10-19T20:48:47.007480+00:00,26.0,2025-10-28T22:21:44.628040+00:00,1.6376033222222224,1.0,1.0,0.01,0.01,133.0,0.01,8542.0,26.0,/historical/trades\n2025-10-29,2025-10-28,FED-25OCT,4.75,FED-25OCT-T4.75,complete,1.0,2025-10-27T00:00:00+00:00,2025-10-29T00:00:00+00:00,2026-10-19T20:48:50.873755+00:00,1.0,2025-10-27T22:19:16.758531+00:00,25.67867818583333,1.0,1.0,0.01,0.01,80.0,0.01,80.0,1.0,/historical/trades\n2025-12-10,2025-12-09,FED-25DEC,3.0,FED-25DEC-T3.00,complete,1.0,2025-12-08T00:00:00+00:00,2025-12-10T00:00:00+00:00,2026-10-19T20:48:50.732820+00:00,1.0,2025-12-09T10:46:39.007937+00:00,13.222497795277777,1.0,1.0,0.99,0.99,66.0,0.99,66.0,1.0,/historical/trades\n2025-12-10,2025-12-09,FED-25DEC,3.5,FED-25DEC-T3.50,complete,1.0,2025-12-08T00:00:00+00:00,2025-12-10T00:00:00+00:00,2026-10-19T20:48:50.755182+00:00,16.0,2025-12-09T23:35:43.514121+00:00,0.4045794108333334,1.0,1.0,0.99,0.99,250.0,0.99,4618.0,16.0,/historical/trades\n2025-12-10,2025-12-09,FED-25DEC,3.75,FED-25DEC-T3.75,complete,1.0,2025-12-08T00:00:00+00:00,2025-12-10T00:00:00+00:00,2026-10-19T20:48:55.276133+00:00,107.0,2025-12-09T23:48:15.074505+00:00,0.1958126375,1.0,1.0,0.04,0.04,4.0,0.04,11591.0,107.0,/historical/trades\n2025-12-10,2025-12-09,FED-25DEC,4.0,FED-25DEC-T4.00,complete,1.0,2025-12-08T00:00:00+00:00,2025-12-10T00:00:00+00:00,2026-10-19T20:48:55.301705+00:00,16.0,2025-12-09T23:39:39.825997+00:00,0.3389372230555555,1.0,1.0,0.01,0.01,21.0,0.01,4184.0,16.0,/historical/trades\n2025-12-10,2025-12-09,FED-25DEC,4.25,FED-25DEC-T4.25,complete,1.0,2025-12-08T00:00:00+00:00,2025-12-10T00:00:00+00:00,2026-10-19T20:48:55.727175+00:00,2.0,2025-12-09T10:57:20.673962+00:00,13.044257232777776,1.0,1.0,0.01,0.01,100.0,0.01,266.0,2.0,/historical/trades\n2025-12-10,2025-12-09,FED-25DEC,4.75,FED-25DEC-T4.75,complete,1.0,2025-12-08T00:00:00+00:00,2025-12-10T00:00:00+00:00,2026-10-19T20:48:59.260174+00:00,2.0,2025-12-09T04:55:12.118215+00:00,19.0799671625,1.0,1.0,0.01,0.01,333.0,0.01,366.0,2.0,/historical/trades\n2025-12-10,2025-12-09,FED-25DEC,5.25,FED-25DEC-T5.25,complete,1.0,2025-12-08T00:00:00+00:00,2025-12-10T00:00:00+00:00,2026-10-19T20:48:59.198373+00:00,1.0,2025-12-08T00:42:49.185632+00:00,47.28633732444444,1.0,1.0,0.01,0.01,51.0,0.01,51.0,1.0,/historical/trades\n2026-01-28,2026-01-27,FED-26JAN,2.75,KXFED-26JAN-T2.75,complete,1.0,2026-01-26T00:00:00+00:00,2026-01-28T00:00:00+00:00,2026-10-19T20:48:59.201082+00:00,4.0,2026-01-27T17:11:52.022105+00:00,6.802216081944445,2.0,1.0,0.99,0.99,500.0,0.99,866.0,4.0,/historical/trades\n2026-01-28,2026-01-27,FED-26JAN,3.0,KXFED-26JAN-T3.00,complete,1.0,2026-01-26T00:00:00+00:00,2026-01-28T00:00:00+00:00,2026-10-19T20:49:03.325308+00:00,6.0,2026-01-27T23:57:38.108535+00:00,0.0394142958333333,6.0,1.0,0.99,0.99,6666.0,0.99,6666.0,6.0,/historical/trades\n2026-01-28,2026-01-27,FED-26JAN,3.25,KXFED-26JAN-T3.25,complete,1.0,2026-01-26T00:00:00+00:00,2026-01-28T00:00:00+00:00,2026-10-19T20:49:03.335501+00:00,5.0,2026-01-27T23:57:14.752316+00:00,0.0459021344444444,5.0,1.0,0.99,0.99,6666.0,0.99,6666.0,5.0,/historical/trades\n2026-01-28,2026-01-27,FED-26JAN,3.5,KXFED-26JAN-T3.50,complete,1.0,2026-01-26T00:00:00+00:00,2026-01-28T00:00:00+00:00,2026-10-19T20:49:03.726666+00:00,54.0,2026-01-27T23:55:18.305194+00:00,0.0782485572222222,2.0,1.0,0.98,0.98,5000.0,0.98,38321.0,54.0,/historical/trades\n2026-01-28,2026-01-27,FED-26JAN,3.75,KXFED-26JAN-T3.75,complete,1.0,2026-01-26T00:00:00+00:00,2026-01-28T00:00:00+00:00,2026-10-19T20:49:07.385909+00:00,155.0,2026-01-27T23:58:50.539356+00:00,0.0192946233333333,1.0,1.0,0.01,0.01,6666.0,0.01,78536.0,155.0,/historical/trades\n2026-01-28,2026-01-27,FED-26JAN,4.0,KXFED-26JAN-T4.00,complete,1.0,2026-01-26T00:00:00+00:00,2026-01-28T00:00:00+00:00,2026-10-19T20:49:07.354165+00:00,82.0,2026-01-27T21:20:44.726363+00:00,2.6542426769444445,1.0,1.0,0.01,0.01,467.0,0.01,55525.0,82.0,/historical/trades\n2026-01-28,2026-01-27,FED-26JAN,4.25,KXFED-26JAN-T4.25,complete,1.0,2026-01-26T00:00:00+00:00,2026-01-28T00:00:00+00:00,2026-10-19T20:49:07.343281+00:00,20.0,2026-01-27T08:57:00.674581+00:00,15.04981261638889,1.0,1.0,0.01,0.01,18.0,0.01,4672.0,20.0,/historical/trades\n2026-01-28,2026-01-27,FED-26JAN,4.5,KXFED-26JAN-T4.50,complete,1.0,2026-01-26T00:00:00+00:00,2026-01-28T00:00:00+00:00,2026-10-19T20:49:11.138848+00:00,5.0,2026-01-27T05:12:15.504678+00:00,18.795693145,1.0,1.0,0.01,0.01,93.0,0.01,991.0,5.0,/historical/trades\n2026-01-28,2026-01-27,FED-26JAN,4.75,KXFED-26JAN-T4.75,complete,1.0,2026-01-26T00:00:00+00:00,2026-01-28T00:00:00+00:00,2026-10-19T20:49:11.143785+00:00,1.0,2026-01-26T23:52:12.128223+00:00,24.1299643825,1.0,1.0,0.01,0.01,93.0,0.01,93.0,1.0,/historical/trades\n2026-01-28,2026-01-27,FED-26JAN,5.25,KXFED-26JAN-T5.25,complete,1.0,2026-01-26T00:00:00+00:00,2026-01-28T00:00:00+00:00,2026-10-19T20:49:11.158990+00:00,1.0,2026-01-26T22:57:45.193436+00:00,25.037446267777774,1.0,1.0,0.01,0.01,1870.0,0.01,1870.0,1.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,2.75,KXFED-26MAR-T2.75,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:15.024931+00:00,6.0,2026-03-17T19:53:01.563988+00:00,4.116232225555556,1.0,1.0,0.99,0.99,93.0,0.99,8796.0,6.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,3.0,KXFED-26MAR-T3.00,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:15.006102+00:00,4.0,2026-03-16T19:31:26.943261+00:00,28.47584909416667,1.0,1.0,0.99,0.99,1.0,0.99,470.0,4.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,3.25,KXFED-26MAR-T3.25,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:14.963710+00:00,12.0,2026-03-17T23:53:07.705602+00:00,0.1145262216666666,1.0,1.0,0.99,0.99,333.0,0.99,13424.0,12.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,3.5,KXFED-26MAR-T3.50,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:18.623147+00:00,29.0,2026-03-17T23:51:18.319067+00:00,0.1449113702777777,1.0,1.0,0.99,0.99,93.0,0.99,15958.0,29.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,3.75,KXFED-26MAR-T3.75,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:18.590233+00:00,317.0,2026-03-17T23:59:10.235848+00:00,0.0138233755555555,1.0,1.0,0.01,0.01,935.0,0.01,214544.0,317.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,4.0,KXFED-26MAR-T4.00,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:18.651429+00:00,77.0,2026-03-17T22:55:17.500731+00:00,1.0784720191666666,1.0,1.0,0.01,0.01,38.0,0.01,45298.0,77.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,4.25,KXFED-26MAR-T4.25,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:22.528967+00:00,17.0,2026-03-17T19:15:33.952921+00:00,4.740568633055555,1.0,1.0,0.01,0.01,1.0,0.01,2804.0,17.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,4.5,KXFED-26MAR-T4.50,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:22.541761+00:00,4.0,2026-03-17T19:15:34.493517+00:00,4.7404184675000005,1.0,1.0,0.01,0.01,1.0,0.01,113.0,4.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,4.75,KXFED-26MAR-T4.75,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:22.591047+00:00,3.0,2026-03-17T19:15:35.066780+00:00,4.740259227777777,1.0,1.0,0.01,0.01,1.0,0.01,95.0,3.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,5.0,KXFED-26MAR-T5.00,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:25.136350+00:00,4.0,2026-03-17T19:15:35.622270+00:00,4.740104925,1.0,1.0,0.01,0.01,1.0,0.01,188.0,4.0,/historical/trades\n2026-03-18,2026-03-17,FED-26MAR,5.25,KXFED-26MAR-T5.25,complete,1.0,2026-03-16T00:00:00+00:00,2026-03-18T00:00:00+00:00,2026-10-19T20:49:25.296587+00:00,6.0,2026-03-17T19:15:36.204020+00:00,4.739943327777778,1.0,1.0,0.01,0.01,1.0,0.01,2151.0,6.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,2.75,KXFED-26APR-T2.75,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:25.132987+00:00,4.0,2026-04-28T19:51:58.711165+00:00,4.133691343055555,1.0,1.0,0.99,0.99,3.0,0.99,199.37,4.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,3.0,KXFED-26APR-T3.00,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:28.823603+00:00,6.0,2026-04-28T20:21:12.703838+00:00,3.646471156111111,1.0,1.0,0.99,0.99,3333.0,0.99,3726.0500000000006,6.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,3.25,KXFED-26APR-T3.25,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:28.831138+00:00,9.0,2026-04-28T22:37:43.994056+00:00,1.3711127622222223,1.0,1.0,0.99,0.99,333.0,0.99,869.31,9.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,3.5,KXFED-26APR-T3.50,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:28.870444+00:00,55.0,2026-04-28T22:51:39.855160+00:00,1.1389291222222222,1.0,1.0,0.99,0.99,2337.66,0.99,11574.2,55.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,3.75,KXFED-26APR-T3.75,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:35.424796+00:00,380.0,2026-04-28T23:59:58.917677+00:00,0.0003006452777777,1.0,1.0,0.01,0.01,100.0,0.01,348554.71,380.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,4.0,KXFED-26APR-T4.00,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:35.430298+00:00,222.0,2026-04-28T23:51:29.725064+00:00,0.1417430377777777,1.0,1.0,0.01,0.01,93.51,0.01,108883.57000000002,222.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,4.25,KXFED-26APR-T4.25,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:35.469118+00:00,77.0,2026-04-28T23:52:40.558234+00:00,0.1220671572222222,1.0,1.0,0.01,0.01,467.59,0.01,26140.849999999995,77.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,4.5,KXFED-26APR-T4.50,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:38.864438+00:00,9.0,2026-04-28T07:03:49.476348+00:00,16.93625657,1.0,1.0,0.01,0.01,93.51,0.01,854.47,9.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,4.75,KXFED-26APR-T4.75,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:38.922604+00:00,4.0,2026-04-28T23:00:14.579467+00:00,0.9959501480555556,1.0,1.0,0.01,0.01,10.0,0.01,1061.7,4.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,5.0,KXFED-26APR-T5.00,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:38.857340+00:00,5.0,2026-04-28T19:04:18.224359+00:00,4.928271011388889,1.0,1.0,0.01,0.01,410.0,0.0099999999999999,646.51,5.0,/historical/trades\n2026-04-29,2026-04-28,FED-26APR,5.25,KXFED-26APR-T5.25,complete,1.0,2026-04-27T00:00:00+00:00,2026-04-29T00:00:00+00:00,2026-10-19T20:49:43.100264+00:00,2.0,2026-04-28T07:04:09.711673+00:00,16.93063564638889,1.0,1.0,0.01,0.01,93.51,0.01,153.51,2.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,2.75,KXFED-26JUN-T2.75,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:49:43.168118+00:00,6.0,2026-06-16T16:11:48.819033+00:00,7.803105824166667,1.0,1.0,0.99,0.99,254.0,0.99,992.17,6.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,3.0,KXFED-26JUN-T3.00,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:49:43.145628+00:00,37.0,2026-06-16T23:45:55.439546+00:00,0.2346001261111111,1.0,1.0,0.99,0.99,83.0,0.99,6063.51,37.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,3.25,KXFED-26JUN-T3.25,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:49:46.320273+00:00,11.0,2026-06-16T16:15:52.381250+00:00,7.735449652777778,1.0,1.0,0.99,0.99,400.0,0.99,11179.1,11.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,3.5,KXFED-26JUN-T3.50,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:49:46.366650+00:00,94.0,2026-06-16T23:17:13.700144+00:00,0.7128610711111112,1.0,1.0,0.99,0.99,16.0,0.99,51300.97,94.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,3.75,KXFED-26JUN-T3.75,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:49:46.426855+00:00,344.0,2026-06-16T23:52:05.765198+00:00,0.1317318894444444,1.0,1.0,0.01,0.01,426.0,0.01,247579.72000000003,344.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,4.0,KXFED-26JUN-T4.00,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:50:03.386747+00:00,110.0,2026-06-16T23:23:01.183727+00:00,0.616337853611111,1.0,1.0,0.01,0.01,654.63,0.01,70589.53,110.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,4.25,KXFED-26JUN-T4.25,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:50:03.321297+00:00,8.0,2026-06-16T17:12:51.451003+00:00,6.785708054722223,1.0,1.0,0.01,0.01,500.0,0.01,3042.51,8.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,4.5,KXFED-26JUN-T4.50,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:50:03.306438+00:00,42.0,2026-06-16T23:37:15.156145+00:00,0.3791232930555556,1.0,1.0,0.01,0.01,83.0,0.01,7692.35,42.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,4.75,KXFED-26JUN-T4.75,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:50:08.009764+00:00,1.0,2026-06-15T22:22:11.935570+00:00,25.63001789722222,1.0,1.0,0.01,0.01,666.33,0.01,666.33,1.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,5.0,KXFED-26JUN-T5.00,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:50:07.770524+00:00,2.0,2026-06-15T22:22:37.273052+00:00,25.622979707777777,1.0,1.0,0.01,0.01,666.33,0.01,1332.66,2.0,/historical/trades\n2026-06-17,2026-06-16,FED-26JUN,5.25,KXFED-26JUN-T5.25,complete,1.0,2026-06-15T00:00:00+00:00,2026-06-17T00:00:00+00:00,2026-10-19T20:50:07.781688+00:00,1.0,2026-06-15T14:00:12.107056+00:00,33.99663692888889,1.0,1.0,0.01,0.01,93.51,0.01,93.51,1.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,2.75,KXFED-26JUL-T2.75,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:12.705371+00:00,4.0,2026-07-28T00:08:08.987049+00:00,23.864170264166667,1.0,1.0,0.99,0.99,69.0,0.99,167.93,4.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,3.0,KXFED-26JUL-T3.00,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:12.741510+00:00,5.0,2026-07-28T00:08:08.862406+00:00,23.864204887222225,1.0,1.0,0.99,0.99,69.0,0.99,246.93,5.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,3.25,KXFED-26JUL-T3.25,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:12.853485+00:00,6.0,2026-07-28T13:17:52.834528+00:00,10.70199040888889,1.0,1.0,0.99,0.99,250.0,0.99,496.93,6.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,3.5,KXFED-26JUL-T3.50,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:15.399313+00:00,18.0,2026-07-28T22:09:40.733948+00:00,1.8386850144444444,3.0,1.0,0.99,0.99,3740.0,0.99,11627.92,18.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,3.75,KXFED-26JUL-T3.75,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:15.474646+00:00,394.0,2026-07-28T23:44:44.531387+00:00,0.2542968369444444,1.0,1.0,0.26,0.26,8.34,0.26,81696.34,394.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,4.0,KXFED-26JUL-T4.00,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:15.531959+00:00,124.0,2026-07-28T23:45:40.410897+00:00,0.2387747508333333,1.0,1.0,0.01,0.01,116.89,0.01,72958.12999999999,124.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,4.25,KXFED-26JUL-T4.25,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:21.224689+00:00,15.0,2026-07-28T22:04:13.653256+00:00,1.9295407622222225,1.0,1.0,0.01,0.01,20.0,0.01,3543.3400000000006,15.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,4.5,KXFED-26JUL-T4.50,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:21.264517+00:00,1.0,2026-07-27T01:00:37.606595+00:00,46.98955372361112,1.0,1.0,0.01,0.01,0.93,0.01,0.93,1.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,4.75,KXFED-26JUL-T4.75,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:21.335027+00:00,12.0,2026-07-28T22:04:05.719507+00:00,1.931744581388889,1.0,1.0,0.01,0.01,25.0,0.01,1074.25,12.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,5.0,KXFED-26JUL-T5.00,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:26.236343+00:00,3.0,2026-07-28T22:04:36.104450+00:00,1.9233043194444448,1.0,1.0,0.01,0.01,20.0,0.01,22.93,3.0,/historical/trades\n2026-07-29,2026-07-28,FED-26JUL,5.25,KXFED-26JUL-T5.25,complete,1.0,2026-07-27T00:00:00+00:00,2026-07-29T00:00:00+00:00,2026-10-19T20:50:26.268575+00:00,1.0,2026-07-27T01:01:11.772471+00:00,46.9800632025,1.0,1.0,0.01,0.01,0.93,0.01,0.93,1.0,/historical/trades\n2026-09-16,2026-09-15,FED-26SEP,2.75,KXFED-26SEP-T2.75,complete,1.0,2026-09-14T00:00:00+00:00,2026-09-16T00:00:00+00:00,2026-10-19T20:51:00.307104+00:00,2.0,2026-09-15T22:16:47.168805+00:00,1.7202308875,1.0,1.0,0.99,0.99,93.51,0.99,393.17,2.0,/markets/trades\n2026-09-16,2026-09-15,FED-26SEP,3.0,KXFED-26SEP-T3.00,complete,1.0,2026-09-14T00:00:00+00:00,2026-09-16T00:00:00+00:00,2026-10-19T20:51:00.314736+00:00,2.0,2026-09-15T17:02:28.134791+00:00,6.958851446944444,1.0,1.0,0.99,0.99,467.59,0.99,647.5899999999999,2.0,/markets/trades\n2026-09-16,2026-09-15,FED-26SEP,3.25,KXFED-26SEP-T3.25,complete,1.0,2026-09-14T00:00:00+00:00,2026-09-16T00:00:00+00:00,2026-10-19T20:51:00.329480+00:00,2.0,2026-09-15T14:28:44.361851+00:00,9.521010596944445,1.0,1.0,0.99,0.99,93.51,0.99,273.51,2.0,/markets/trades\n2026-09-16,2026-09-15,FED-26SEP,3.5,KXFED-26SEP-T3.50,complete,1.0,2026-09-14T00:00:00+00:00,2026-09-16T00:00:00+00:00,2026-10-19T20:51:04.151588+00:00,22.0,2026-09-15T22:49:01.780025+00:00,1.1828388819444444,1.0,1.0,0.99,0.99,467.0,0.99,20405.74,22.0,/markets/trades\n2026-09-16,2026-09-15,FED-26SEP,3.75,KXFED-26SEP-T3.75,complete,1.0,2026-09-14T00:00:00+00:00,2026-09-16T00:00:00+00:00,2026-10-19T20:51:04.337277+00:00,797.0,2026-09-15T23:52:18.892099+00:00,0.1280855280555555,1.0,1.0,0.88,0.88,27.0,0.88,154784.44999999998,797.0,/markets/trades\n2026-09-16,2026-09-15,FED-26SEP,4.0,KXFED-26SEP-T4.00,complete,1.0,2026-09-14T00:00:00+00:00,2026-09-16T00:00:00+00:00,2026-10-19T20:51:04.167367+00:00,200.0,2026-09-15T23:46:08.594289+00:00,0.2309460308333333,1.0,1.0,0.01,0.01,8000.0,0.01,262749.27,200.0,/markets/trades\n2026-09-16,2026-09-15,FED-26SEP,4.25,KXFED-26SEP-T4.25,complete,1.0,2026-09-14T00:00:00+00:00,2026-09-16T00:00:00+00:00,2026-10-19T20:51:07.859573+00:00,5.0,2026-09-15T18:31:46.897507+00:00,5.470306248055556,1.0,1.0,0.01,0.01,1356.02,0.01,2882.17,5.0,/markets/trades\n2026-09-16,2026-09-15,FED-26SEP,5.0,KXFED-26SEP-T5.00,complete,1.0,2026-09-14T00:00:00+00:00,2026-09-16T00:00:00+00:00,2026-10-19T20:51:07.848129+00:00,4.0,2026-09-15T14:43:30.254000+00:00,9.274929444444444,1.0,1.0,0.01,0.01,200.0,0.01,3943.33,4.0,/markets/trades\n', 'authors_bins_csv': 'event_key,date,strike,probability\nFED-22DEC,2022-12-13,0.0,0.0\nFED-22JAN,2022-01-25,0.0,96.93877551020408\nFED-22JUL,2022-07-26,0.0,0.0\nFED-22JUN,2022-06-14,0.0,0.0\nFED-22MAR,2022-03-15,0.0,1.020408163265306\nFED-22MAY,2022-05-03,0.0,0.0\nFED-22NOV,2022-11-01,0.0,0.0\nFED-22SEP,2022-09-20,0.0,0.0\nFED-22DEC,2022-12-13,0.25,0.0\nFED-22JAN,2022-01-25,0.25,3.061224489795918\nFED-22JUL,2022-07-26,0.25,0.0\nFED-22JUN,2022-06-14,0.25,0.0\nFED-22MAR,2022-03-15,0.25,93.87755102040816\nFED-22MAY,2022-05-03,0.25,0.0\nFED-22NOV,2022-11-01,0.25,0.0\nFED-22SEP,2022-09-20,0.25,0.0\nFED-22DEC,2022-12-13,0.5,0.0\nFED-22JUL,2022-07-26,0.5,0.0\nFED-22JUN,2022-06-14,0.5,0.0\nFED-22MAR,2022-03-15,0.5,4.081632653061225\nFED-22MAY,2022-05-03,0.5,6.122448979591836\nFED-22NOV,2022-11-01,0.5,0.0\nFED-22SEP,2022-09-20,0.5,0.0\nFED-22DEC,2022-12-13,0.75,0.0\nFED-22JUL,2022-07-26,0.75,0.0\nFED-22JUN,2022-06-14,0.75,0.0\nFED-22MAR,2022-03-15,0.75,1.020408163265306\nFED-22MAY,2022-05-03,0.75,89.79591836734694\nFED-22NOV,2022-11-01,0.75,0.0\nFED-22SEP,2022-09-20,0.75,0.0\nFED-22DEC,2022-12-13,1.0,0.0\nFED-22JUL,2022-07-26,1.0,0.0\nFED-22JUN,2022-06-14,1.0,2.0408163265306123\nFED-22MAY,2022-05-03,1.0,4.081632653061225\nFED-22NOV,2022-11-01,1.0,0.0\nFED-22SEP,2022-09-20,1.0,0.0\nFED-22DEC,2022-12-13,1.25,0.0\nFED-22JUL,2022-07-26,1.25,0.0\nFED-22JUN,2022-06-14,1.25,17.346938775510203\nFED-22NOV,2022-11-01,1.25,0.0\nFED-22SEP,2022-09-20,1.25,0.0\nFED-22DEC,2022-12-13,1.5,0.0\nFED-22JUL,2022-07-26,1.5,0.0\nFED-22JUN,2022-06-14,1.5,75.51020408163265\nFED-22NOV,2022-11-01,1.5,0.0\nFED-22SEP,2022-09-20,1.5,0.0\nFED-23FEB,2023-01-31,1.5,0.0\nFED-22DEC,2022-12-13,1.75,0.0\nFED-22JUL,2022-07-26,1.75,0.0\nFED-22JUN,2022-06-14,1.75,5.1020408163265305\nFED-22NOV,2022-11-01,1.75,0.0\nFED-22SEP,2022-09-20,1.75,0.0\nFED-23FEB,2023-01-31,1.75,0.0\nFED-22DEC,2022-12-13,2.0,0.0\nFED-22JUL,2022-07-26,2.0,2.0408163265306123\nFED-22NOV,2022-11-01,2.0,0.0\nFED-22SEP,2022-09-20,2.0,0.0\nFED-23FEB,2023-01-31,2.0,0.0\nFED-22DEC,2022-12-13,2.25,0.0\nFED-22JUL,2022-07-26,2.25,88.77551020408163\nFED-22NOV,2022-11-01,2.25,0.0\nFED-22SEP,2022-09-20,2.25,0.0\nFED-23FEB,2023-01-31,2.25,0.0\nFED-22DEC,2022-12-13,2.5,0.0\nFED-22JUL,2022-07-26,2.5,9.183673469387756\nFED-22NOV,2022-11-01,2.5,0.0\nFED-22SEP,2022-09-20,2.5,0.0\nFED-23FEB,2023-01-31,2.5,0.0\nFED-25DEC,2025-12-09,2.5,0.0\nFED-25JUL,2025-07-29,2.5,0.0\nFED-25JUN,2025-06-17,2.5,0.0\nFED-25MAR,2025-03-18,2.5,0.0\nFED-25MAY,2025-05-06,2.5,0.0\nFED-25OCT,2025-10-28,2.5,0.0\nFED-25SEP,2025-09-16,2.5,0.0\nFED-26APR,2026-04-28,2.5,0.0\nFED-26JAN,2026-01-27,2.5,0.0\nFED-26JUL,2026-07-28,2.5,0.0\nFED-26JUN,2026-06-16,2.5,0.0\nFED-26MAR,2026-03-17,2.5,0.0\nFED-26SEP,2026-09-15,2.5,0.0\nFED-22DEC,2022-12-13,2.75,0.0\nFED-22JUL,2022-07-26,2.75,0.0\nFED-22NOV,2022-11-01,2.75,0.0\nFED-22SEP,2022-09-20,2.75,1.020408163265306\nFED-23FEB,2023-01-31,2.75,0.0\nFED-23JUL,2023-07-25,2.75,0.0\nFED-23JUN,2023-06-13,2.75,0.0\nFED-23MAR,2023-03-21,2.75,0.0\nFED-23MAY,2023-05-02,2.75,0.0\nFED-23NOV,2023-10-31,2.75,0.0\nFED-23SEP,2023-09-19,2.75,0.0\nFED-25DEC,2025-12-09,2.75,0.0\nFED-25JUL,2025-07-29,2.75,0.0\nFED-25JUN,2025-06-17,2.75,0.0\nFED-25MAR,2025-03-18,2.75,0.0\nFED-25MAY,2025-05-06,2.75,0.0\nFED-25OCT,2025-10-28,2.75,0.0\nFED-25SEP,2025-09-16,2.75,0.0\nFED-26APR,2026-04-28,2.75,0.0\nFED-26JAN,2026-01-27,2.75,0.0\nFED-26JUL,2026-07-28,2.75,0.0\nFED-26JUN,2026-06-16,2.75,0.0\nFED-26MAR,2026-03-17,2.75,0.0\nFED-26SEP,2026-09-15,2.75,0.0\nFED-22DEC,2022-12-13,3.0,0.0\nFED-22NOV,2022-11-01,3.0,0.0\nFED-22SEP,2022-09-20,3.0,92.85714285714286\nFED-23FEB,2023-01-31,3.0,0.0\nFED-23JUL,2023-07-25,3.0,0.0\nFED-23JUN,2023-06-13,3.0,0.0\nFED-23MAR,2023-03-21,3.0,0.0\nFED-23MAY,2023-05-02,3.0,0.0\nFED-23NOV,2023-10-31,3.0,0.0\nFED-23SEP,2023-09-19,3.0,0.0\nFED-24DEC,2024-12-17,3.0,0.0\nFED-25DEC,2025-12-09,3.0,0.0\nFED-25JAN,2025-01-28,3.0,0.0\nFED-25JUL,2025-07-29,3.0,0.0\nFED-25JUN,2025-06-17,3.0,0.0\nFED-25MAR,2025-03-18,3.0,0.0\nFED-25MAY,2025-05-06,3.0,0.0\nFED-25OCT,2025-10-28,3.0,0.0\nFED-25SEP,2025-09-16,3.0,0.0\nFED-26APR,2026-04-28,3.0,0.0\nFED-26JAN,2026-01-27,3.0,0.0\nFED-26JUL,2026-07-28,3.0,0.0\nFED-26JUN,2026-06-16,3.0,0.0\nFED-26MAR,2026-03-17,3.0,0.0\nFED-26SEP,2026-09-15,3.0,0.0\nFED-22DEC,2022-12-13,3.25,0.0\nFED-22NOV,2022-11-01,3.25,0.0\nFED-22SEP,2022-09-20,3.25,6.122448979591838\nFED-23FEB,2023-01-31,3.25,0.0\nFED-23JUL,2023-07-25,3.25,0.0\nFED-23JUN,2023-06-13,3.25,0.0\nFED-23MAR,2023-03-21,3.25,0.0\nFED-23MAY,2023-05-02,3.25,0.0\nFED-23NOV,2023-10-31,3.25,0.0\nFED-23SEP,2023-09-19,3.25,0.0\nFED-24DEC,2024-12-17,3.25,0.0\nFED-24NOV,2024-11-06,3.25,0.0\nFED-25DEC,2025-12-09,3.25,0.0\nFED-25JAN,2025-01-28,3.25,0.0\nFED-25JUL,2025-07-29,3.25,0.0\nFED-25JUN,2025-06-17,3.25,0.0\nFED-25MAR,2025-03-18,3.25,0.0\nFED-25MAY,2025-05-06,3.25,0.0\nFED-25OCT,2025-10-28,3.25,0.0\nFED-25SEP,2025-09-16,3.25,0.0\nFED-26APR,2026-04-28,3.25,0.0\nFED-26JAN,2026-01-27,3.25,1.020408163265306\nFED-26JUL,2026-07-28,3.25,0.0\nFED-26JUN,2026-06-16,3.25,0.0\nFED-26MAR,2026-03-17,3.25,0.0\nFED-26SEP,2026-09-15,3.25,0.0\nFED-22DEC,2022-12-13,3.5,0.0\nFED-22NOV,2022-11-01,3.5,4.081632653061225\nFED-22SEP,2022-09-20,3.5,0.0\nFED-23FEB,2023-01-31,3.5,0.0\nFED-23JUL,2023-07-25,3.5,0.0\nFED-23JUN,2023-06-13,3.5,0.0\nFED-23MAR,2023-03-21,3.5,0.0\nFED-23MAY,2023-05-02,3.5,0.0\nFED-23NOV,2023-10-31,3.5,0.0\nFED-23SEP,2023-09-19,3.5,0.0\nFED-24DEC,2024-12-17,3.5,0.0\nFED-24NOV,2024-11-06,3.5,0.0\nFED-24SEP,2024-09-17,3.5,0.0\nFED-25DEC,2025-12-09,3.5,96.93877551020408\nFED-25JAN,2025-01-28,3.5,0.0\nFED-25JUL,2025-07-29,3.5,0.0\nFED-25JUN,2025-06-17,3.5,0.0\nFED-25MAR,2025-03-18,3.5,0.0\nFED-25MAY,2025-05-06,3.5,0.0\nFED-25OCT,2025-10-28,3.5,1.020408163265306\nFED-25SEP,2025-09-16,3.5,0.0\nFED-26APR,2026-04-28,3.5,100.0\nFED-26JAN,2026-01-27,3.5,98.9795918367347\nFED-26JUL,2026-07-28,3.5,74.48979591836735\nFED-26JUN,2026-06-16,3.5,100.0\nFED-26MAR,2026-03-17,3.5,100.0\nFED-26SEP,2026-09-15,3.5,11.224489795918368\nFED-22DEC,2022-12-13,3.75,0.0\nFED-22NOV,2022-11-01,3.75,94.89795918367346\nFED-23FEB,2023-01-31,3.75,0.0\nFED-23JUL,2023-07-25,3.75,0.0\nFED-23JUN,2023-06-13,3.75,0.0\nFED-23MAR,2023-03-21,3.75,0.0\nFED-23MAY,2023-05-02,3.75,0.0\nFED-23NOV,2023-10-31,3.75,0.0\nFED-23SEP,2023-09-19,3.75,0.0\nFED-24DEC,2024-12-17,3.75,0.0\nFED-24JUL,2024-07-30,3.75,0.0\nFED-24JUN,2024-06-11,3.75,0.0\nFED-24NOV,2024-11-06,3.75,0.0\nFED-24SEP,2024-09-17,3.75,0.0\nFED-25DEC,2025-12-09,3.75,3.061224489795918\nFED-25JAN,2025-01-28,3.75,0.0\nFED-25JUL,2025-07-29,3.75,0.0\nFED-25JUN,2025-06-17,3.75,0.0\nFED-25MAR,2025-03-18,3.75,0.0\nFED-25MAY,2025-05-06,3.75,0.0\nFED-25OCT,2025-10-28,3.75,97.95918367346938\nFED-25SEP,2025-09-16,3.75,5.1020408163265305\nFED-26APR,2026-04-28,3.75,0.0\nFED-26JAN,2026-01-27,3.75,0.0\nFED-26JUL,2026-07-28,3.75,25.51020408163265\nFED-26JUN,2026-06-16,3.75,0.0\nFED-26MAR,2026-03-17,3.75,0.0\nFED-26SEP,2026-09-15,3.75,88.77551020408163\nFED-22DEC,2022-12-13,4.0,13.26530612244898\nFED-22NOV,2022-11-01,4.0,1.020408163265306\nFED-23FEB,2023-01-31,4.0,0.0\nFED-23JUL,2023-07-25,4.0,0.0\nFED-23JUN,2023-06-13,4.0,0.0\nFED-23MAR,2023-03-21,4.0,0.0\nFED-23MAY,2023-05-02,4.0,0.0\nFED-23NOV,2023-10-31,4.0,0.0\nFED-23SEP,2023-09-19,4.0,0.0\nFED-24DEC,2024-12-17,4.0,5.1020408163265305\nFED-24JUL,2024-07-30,4.0,0.0\nFED-24JUN,2024-06-11,4.0,0.0\nFED-24MAY,2024-04-30,4.0,0.0\nFED-24NOV,2024-11-06,4.0,0.0\nFED-24SEP,2024-09-17,4.0,0.0\nFED-25DEC,2025-12-09,4.0,0.0\nFED-25JAN,2025-01-28,4.0,0.0\nFED-25JUL,2025-07-29,4.0,2.0408163265306123\nFED-25JUN,2025-06-17,4.0,1.020408163265306\nFED-25MAR,2025-03-18,4.0,0.0\nFED-25MAY,2025-05-06,4.0,2.0408163265306123\nFED-25OCT,2025-10-28,4.0,1.020408163265306\nFED-25SEP,2025-09-16,4.0,92.85714285714286\nFED-26APR,2026-04-28,4.0,0.0\nFED-26JAN,2026-01-27,4.0,0.0\nFED-26JUL,2026-07-28,4.0,0.0\nFED-26JUN,2026-06-16,4.0,0.0\nFED-26MAR,2026-03-17,4.0,0.0\nFED-26SEP,2026-09-15,4.0,0.0\nFED-22DEC,2022-12-13,4.25,86.73469387755102\nFED-23FEB,2023-01-31,4.25,1.020408163265306\nFED-23JUL,2023-07-25,4.25,0.0\nFED-23JUN,2023-06-13,4.25,0.0\nFED-23MAR,2023-03-21,4.25,0.0\nFED-23MAY,2023-05-02,4.25,0.0\nFED-23NOV,2023-10-31,4.25,0.0\nFED-23SEP,2023-09-19,4.25,0.0\nFED-24DEC,2024-12-17,4.25,91.83673469387756\nFED-24JUL,2024-07-30,4.25,0.0\nFED-24JUN,2024-06-11,4.25,0.0\nFED-24MAR,2024-03-19,4.25,0.0\nFED-24MAY,2024-04-30,4.25,0.0\nFED-24NOV,2024-11-06,4.25,0.0\nFED-24SEP,2024-09-17,4.25,0.0\nFED-25DEC,2025-12-09,4.25,0.0\nFED-25JAN,2025-01-28,4.25,87.75510204081633\nFED-25JUL,2025-07-29,4.25,97.95918367346938\nFED-25JUN,2025-06-17,4.25,98.9795918367347\nFED-25MAR,2025-03-18,4.25,98.9795918367347\nFED-25MAY,2025-05-06,4.25,97.95918367346938\nFED-25OCT,2025-10-28,4.25,0.0\nFED-25SEP,2025-09-16,4.25,2.0408163265306123\nFED-26APR,2026-04-28,4.25,0.0\nFED-26JAN,2026-01-27,4.25,0.0\nFED-26JUL,2026-07-28,4.25,0.0\nFED-26JUN,2026-06-16,4.25,0.0\nFED-26MAR,2026-03-17,4.25,0.0\nFED-26SEP,2026-09-15,4.25,0.0\nFED-22DEC,2022-12-13,4.5,0.0\nFED-23FEB,2023-01-31,4.5,97.95918367346938\nFED-23JUL,2023-07-25,4.5,0.0\nFED-23JUN,2023-06-13,4.5,0.0\nFED-23MAR,2023-03-21,4.5,17.346938775510203\nFED-23MAY,2023-05-02,4.5,0.0\nFED-23NOV,2023-10-31,4.5,0.0\nFED-23SEP,2023-09-19,4.5,0.0\nFED-24DEC,2024-12-17,4.5,1.020408163265306\nFED-24JUL,2024-07-30,4.5,0.0\nFED-24JUN,2024-06-11,4.5,0.0\nFED-24MAR,2024-03-19,4.5,0.0\nFED-24MAY,2024-04-30,4.5,0.0\nFED-24NOV,2024-11-06,4.5,96.93877551020408\nFED-24SEP,2024-09-17,4.5,0.0\nFED-25DEC,2025-12-09,4.5,0.0\nFED-25JAN,2025-01-28,4.5,12.244897959183673\nFED-25JUL,2025-07-29,4.5,0.0\nFED-25JUN,2025-06-17,4.5,0.0\nFED-25MAR,2025-03-18,4.5,1.020408163265306\nFED-25MAY,2025-05-06,4.5,0.0\nFED-25OCT,2025-10-28,4.5,0.0\nFED-25SEP,2025-09-16,4.5,0.0\nFED-26APR,2026-04-28,4.5,0.0\nFED-26JAN,2026-01-27,4.5,0.0\nFED-26JUL,2026-07-28,4.5,0.0\nFED-26JUN,2026-06-16,4.5,0.0\nFED-26MAR,2026-03-17,4.5,0.0\nFED-26SEP,2026-09-15,4.5,0.0\nFED-22DEC,2022-12-13,4.75,0.0\nFED-23FEB,2023-01-31,4.75,1.020408163265306\nFED-23JUL,2023-07-25,4.75,0.0\nFED-23JUN,2023-06-13,4.75,0.0\nFED-23MAR,2023-03-21,4.75,79.59183673469387\nFED-23MAY,2023-05-02,4.75,6.122448979591836\nFED-23NOV,2023-10-31,4.75,0.0\nFED-23SEP,2023-09-19,4.75,0.0\nFED-24DEC,2024-12-17,4.75,2.0408163265306123\nFED-24JAN,2024-01-30,4.75,0.0\nFED-24JUL,2024-07-30,4.75,0.0\nFED-24JUN,2024-06-11,4.75,0.0\nFED-24MAR,2024-03-19,4.75,0.0\nFED-24MAY,2024-04-30,4.75,0.0\nFED-24NOV,2024-11-06,4.75,3.061224489795918\nFED-24SEP,2024-09-17,4.75,53.06122448979592\nFED-25DEC,2025-12-09,4.75,0.0\nFED-25JAN,2025-01-28,4.75,0.0\nFED-25JUL,2025-07-29,4.75,0.0\nFED-25JUN,2025-06-17,4.75,0.0\nFED-25MAR,2025-03-18,4.75,0.0\nFED-25MAY,2025-05-06,4.75,0.0\nFED-25OCT,2025-10-28,4.75,0.0\nFED-25SEP,2025-09-16,4.75,0.0\nFED-26APR,2026-04-28,4.75,0.0\nFED-26JAN,2026-01-27,4.75,0.0\nFED-26JUL,2026-07-28,4.75,0.0\nFED-26JUN,2026-06-16,4.75,0.0\nFED-26MAR,2026-03-17,4.75,0.0\nFED-26SEP,2026-09-15,4.75,0.0\nFED-23FEB,2023-01-31,5.0,0.0\nFED-23JUL,2023-07-25,5.0,0.0\nFED-23JUN,2023-06-13,5.0,95.91836734693878\nFED-23MAR,2023-03-21,5.0,3.061224489795918\nFED-23MAY,2023-05-02,5.0,93.87755102040816\nFED-23NOV,2023-10-31,5.0,0.0\nFED-23SEP,2023-09-19,5.0,0.0\nFED-24DEC,2024-12-17,5.0,0.0\nFED-24JAN,2024-01-30,5.0,0.0\nFED-24JUL,2024-07-30,5.0,1.020408163265306\nFED-24JUN,2024-06-11,5.0,2.0408163265306123\nFED-24MAR,2024-03-19,5.0,0.0\nFED-24MAY,2024-04-30,5.0,3.061224489795918\nFED-24NOV,2024-11-06,5.0,0.0\nFED-24SEP,2024-09-17,5.0,46.93877551020408\nFED-25DEC,2025-12-09,5.0,0.0\nFED-25JAN,2025-01-28,5.0,0.0\nFED-25JUL,2025-07-29,5.0,0.0\nFED-25JUN,2025-06-17,5.0,0.0\nFED-25MAY,2025-05-06,5.0,0.0\nFED-25OCT,2025-10-28,5.0,0.0\nFED-25SEP,2025-09-16,5.0,0.0\nFED-26APR,2026-04-28,5.0,0.0\nFED-26JAN,2026-01-27,5.0,0.0\nFED-26JUL,2026-07-28,5.0,0.0\nFED-26JUN,2026-06-16,5.0,0.0\nFED-26MAR,2026-03-17,5.0,0.0\nFED-26SEP,2026-09-15,5.0,0.0\nFED-23FEB,2023-01-31,5.25,0.0\nFED-23JUL,2023-07-25,5.25,98.9795918367347\nFED-23JUN,2023-06-13,5.25,4.081632653061225\nFED-23MAR,2023-03-21,5.25,0.0\nFED-23MAY,2023-05-02,5.25,0.0\nFED-23NOV,2023-10-31,5.25,100.0\nFED-23SEP,2023-09-19,5.25,98.9795918367347\nFED-24DEC,2024-12-17,5.25,0.0\nFED-24JAN,2024-01-30,5.25,97.95918367346938\nFED-24JUL,2024-07-30,5.25,98.9795918367347\nFED-24JUN,2024-06-11,5.25,97.95918367346938\nFED-24MAR,2024-03-19,5.25,100.0\nFED-24MAY,2024-04-30,5.25,96.93877551020408\nFED-24NOV,2024-11-06,5.25,0.0\nFED-24SEP,2024-09-17,5.25,0.0\nFED-25DEC,2025-12-09,5.25,0.0\nFED-25JAN,2025-01-28,5.25,0.0\nFED-25JUL,2025-07-29,5.25,0.0\nFED-25JUN,2025-06-17,5.25,0.0\nFED-25MAY,2025-05-06,5.25,0.0\nFED-25OCT,2025-10-28,5.25,0.0\nFED-25SEP,2025-09-16,5.25,0.0\nFED-26APR,2026-04-28,5.25,0.0\nFED-26JAN,2026-01-27,5.25,0.0\nFED-26JUL,2026-07-28,5.25,0.0\nFED-26JUN,2026-06-16,5.25,0.0\nFED-26MAR,2026-03-17,5.25,0.0\nFED-26SEP,2026-09-15,5.25,0.0\nFED-23JUL,2023-07-25,5.5,1.020408163265306\nFED-23JUN,2023-06-13,5.5,0.0\nFED-23MAR,2023-03-21,5.5,0.0\nFED-23MAY,2023-05-02,5.5,0.0\nFED-23NOV,2023-10-31,5.5,0.0\nFED-23SEP,2023-09-19,5.5,1.020408163265306\nFED-24DEC,2024-12-17,5.5,0.0\nFED-24JAN,2024-01-30,5.5,2.0408163265306123\nFED-24JUL,2024-07-30,5.5,0.0\nFED-24JUN,2024-06-11,5.5,0.0\nFED-24MAR,2024-03-19,5.5,0.0\nFED-24MAY,2024-04-30,5.5,0.0\nFED-24NOV,2024-11-06,5.5,0.0\nFED-24SEP,2024-09-17,5.5,0.0\nFED-25JAN,2025-01-28,5.5,0.0\nFED-23JUL,2023-07-25,5.75,0.0\nFED-23JUN,2023-06-13,5.75,0.0\nFED-23MAR,2023-03-21,5.75,0.0\nFED-23MAY,2023-05-02,5.75,0.0\nFED-23NOV,2023-10-31,5.75,0.0\nFED-23SEP,2023-09-19,5.75,0.0\nFED-24JAN,2024-01-30,5.75,0.0\nFED-24MAR,2024-03-19,5.75,0.0\nFED-24MAY,2024-04-30,5.75,0.0\nFED-23JUL,2023-07-25,6.0,0.0\nFED-23JUN,2023-06-13,6.0,0.0\nFED-23MAR,2023-03-21,6.0,0.0\nFED-23MAY,2023-05-02,6.0,0.0\nFED-23NOV,2023-10-31,6.0,0.0\nFED-23SEP,2023-09-19,6.0,0.0\nFED-24JAN,2024-01-30,6.0,0.0\nFED-24MAR,2024-03-19,6.0,0.0\nFED-23JUN,2023-06-13,6.25,0.0\nFED-23MAR,2023-03-21,6.25,0.0\nFED-23MAY,2023-05-02,6.25,0.0\nFED-23NOV,2023-10-31,6.25,0.0\nFED-23SEP,2023-09-19,6.25,0.0\nFED-24JAN,2024-01-30,6.25,0.0\nFED-23JUN,2023-06-13,6.5,0.0\nFED-23MAR,2023-03-21,6.5,0.0\nFED-23MAY,2023-05-02,6.5,0.0\nFED-23SEP,2023-09-19,6.5,0.0\n'}
EXPECTED_SHA256 = {'meetings_csv': 'abc1b2f237dd72c920725baba8bb6549b22c652feb047a60b134f900689dedab', 'trades_csv': '8dc418e68c76dd39055a988dd58c321a3877553267460669688fd2efeb32d6ac', 'authors_bins_csv': '48e91ed9a1f74c697d93c88e91ef88df06cdffcfcda19afaae6a6651a9d1679a'}
for key, value in SNAPSHOTS.items():
    assert hashlib.sha256(value.encode()).hexdigest() == EXPECTED_SHA256[key], key
meetings = pd.read_csv(io.StringIO(SNAPSHOTS['meetings_csv']))
trades = pd.read_csv(io.StringIO(SNAPSHOTS['trades_csv']))
authors_bins = pd.read_csv(io.StringIO(SNAPSHOTS['authors_bins_csv']))
assert len(meetings) == 37 and meetings.event_key.is_unique
assert not trades.duplicated(['event_key', 'threshold_pct']).any()
assert trades.status.eq('complete').all(), 'Resolve incomplete source queries before fitting.'
assert trades.age_hours.between(0, 48, inclusive='both').all()
assert trades.latest_batch_vwap.between(0, 1).all()
print('Verified snapshot:', len(meetings), 'meetings;', len(trades), 'threshold summaries')
print('Latest batches with different simultaneous prices:', int(trades.latest_batch_distinct_prices.gt(1).sum()))
display(trades.groupby('status').size().rename('thresholds').to_frame())


### Rules fixed before the revised regressions
- Cutoff: end of the UTC calendar day before the announcement. Query the preceding 48 hours completely using the historical or current trades endpoint as appropriate, paginate, deduplicate trade IDs, then find the latest timestamp. Within a tied latest timestamp, use contract-volume-weighted price. Prices are transaction observations, not executable quotes.
- Legacy contracts are matched only when their settlement rule explicitly says “greater than” the ticker's numerical threshold. Candidate thresholds come from positive reported daily volume on either of the final two UTC days in the authors' source; this is a source-screened reconstruction, not an independent census of all possible markets.
- Primary age limit: 48 hours. Separate stricter sensitivity: 24 hours. Apply equal-weight decreasing isotonic regression to fresh threshold prices. No nearest-49-cent anchor and no inward movement of probability masses.
- Require a **contiguous 25 bp threshold band** with fitted probability at least 99% at its lower threshold and at most 1% at its upper threshold. Choose the narrowest qualifying band; ties favor more central probability, then the lower threshold. Do not bridge missing thresholds.
- Difference adjacent fitted probabilities, put each mass at the next 25 bp upper-bound rate, and normalize within the band. This assumes a 25 bp policy grid. The resulting standard deviation is **conditional on that central band**, not full-distribution dispersion. The omitted implied tail mass is at most 2%; its contribution to full variance is unbounded without further assumptions.
- Compare three measures on identical eligible meetings: fresh conditional dispersion; authors' final probabilities conditional on the same band; original authors' full dispersion. The latter comparison also changes the definition of dispersion. Membership can select more liquid meetings.


In [ ]:
rows=[];points=[]
for hours in [48,24]:
 for _,e in meetings.iterrows():
  g=trades[(trades.event_key==e.event_key)&(trades.status=='complete')&(trades.age_hours<=hours)].sort_values('threshold_pct').copy()
  r=dict(meeting_date=e.meeting_date,event_key=e.event_key,max_age_hours=hours,fresh_thresholds=len(g),eligible=False,reason='fewer than two fresh thresholds')
  if len(g)>=2:
   k=g.threshold_pct.to_numpy();raw=g.latest_batch_vwap.to_numpy();q=isotonic_regression(raw,increasing=False).x
   r['max_isotonic_change_pp']=100*max(abs(q-raw))
   choices=[]
   for i in range(len(k)-1):
    for j in range(i+1,len(k)):
     if np.allclose(np.diff(k[i:j+1]),.25) and q[i]>=.99-1e-10 and q[j]<=.01+1e-10:
      choices.append((k[j]-k[i],-(q[i]-q[j]),k[i],i,j))
   r['reason']='no contiguous 25bp bracket with each implied tail at most 1%'
   if choices:
    _,_,_,i,j=min(choices);w=q[i:j]-q[i+1:j+1];mass=w.sum();p=w/mass;x=k[i+1:j+1];mean=p@x
    r.update(eligible=True,reason='eligible',lower_threshold_pct=k[i],upper_threshold_pct=k[j],central_implied_mass=mass,fresh_central_sigma_bps=100*np.sqrt(p@((x-mean)**2)))
    old=authors_bins[(authors_bins.event_key==e.event_key)&(authors_bins.date==e.observation_date)].copy();old['upper']=old.strike+.25
    old=old[(old.upper>k[i]+1e-8)&(old.upper<=k[j]+1e-8)]
    op=old.probability.to_numpy()/100;ox=old.upper.to_numpy();om=op.sum()
    r['authors_same_band_mass']=om
    if om>0:
     op=op/om;r['authors_same_band_sigma_bps']=100*np.sqrt(op@((ox-op@ox)**2))
   for z in range(len(g)):
    points.append(dict(meeting_date=e.meeting_date,max_age_hours=hours,threshold_pct=k[z],raw_price=raw[z],isotonic_price=q[z],age_hours=g.age_hours.iloc[z]))
  rows.append(r)

coverage = pd.DataFrame(rows).merge(meetings, on=['meeting_date', 'event_key'], validate='many_to_one')
points = pd.DataFrame(points)
coverage.to_csv(OUT / 'meeting_coverage.csv', index=False)
points.to_csv(OUT / 'fresh_threshold_adjustments.csv', index=False)
trades.to_csv(OUT / 'derived_trade_summaries.csv', index=False)
display(coverage.groupby('max_age_hours').eligible.agg(['sum', 'count']).rename(columns={'sum':'eligible', 'count':'candidates'}))
display(coverage[['meeting_date','max_age_hours','fresh_thresholds','eligible','reason']])


### Fixed models, matched samples
A: absolute 2-year Treasury statement-window move ~ dispersion.

B0: the same outcome ~ six-month monetary-policy uncertainty. B1 adds dispersion to B0, using exactly the same rows. Coefficients are yield-move basis points per dispersion basis point. Confidence intervals use HC3 standard errors and a t reference distribution. All fits are exploratory, in sample, and especially fragile when the eligible sample is small. No specification is selected by its p-value. This check does not rebuild the five-day revision measure.


In [ ]:
results = []
for hours in [48, 24]:
    sample = coverage.loc[coverage.max_age_hours.eq(hours) & coverage.eligible].copy()
    assert sample[['abs_ust2y_bps','mpu6m_bps','fresh_central_sigma_bps','kalshi_sigma_bps','authors_same_band_sigma_bps']].notna().all().all()
    for label, variable in [('fresh central', 'fresh_central_sigma_bps'), ('authors same band', 'authors_same_band_sigma_bps'), ('authors full', 'kalshi_sigma_bps')]:
        for model, predictors in [('A', [variable]), ('B1', ['mpu6m_bps', variable])]:
            X = sm.add_constant(sample[predictors], has_constant='add')
            if len(sample) < 10 or np.linalg.matrix_rank(X) < X.shape[1]:
                results.append(dict(max_age_hours=hours, measure=label, model=model, n=len(sample), status='insufficient observations or rank'))
                continue
            fit = sm.OLS(sample.abs_ust2y_bps, X, missing='raise').fit(cov_type='HC3', use_t=True)
            base_X = sm.add_constant(sample[['mpu6m_bps']], has_constant='add')
            base = sm.OLS(sample.abs_ust2y_bps, base_X, missing='raise').fit()
            ci = fit.conf_int().loc[variable]
            results.append(dict(max_age_hours=hours, measure=label, model=model, n=len(sample), status='exploratory', beta=fit.params[variable], ci_low=ci.iloc[0], ci_high=ci.iloc[1], p=fit.pvalues[variable], r2=fit.rsquared, delta_r2_vs_benchmark=fit.rsquared-base.rsquared if model=='B1' else np.nan))
results = pd.DataFrame(results)
results.to_csv(OUT / 'matched_sample_models.csv', index=False)
display(results.round(4))


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
for ax, hours in zip(axes, [48,24]):
    s = coverage.loc[coverage.max_age_hours.eq(hours) & coverage.eligible]
    ax.scatter(s.kalshi_sigma_bps, s.fresh_central_sigma_bps, color='#176b87')
    end = max(s.kalshi_sigma_bps.max(), s.fresh_central_sigma_bps.max()) * 1.1
    ax.plot([0,end],[0,end], color='grey', linestyle='--', linewidth=1)
    ax.set(xlabel='Authors full dispersion (bp)', ylabel='Fresh central-band dispersion (bp)', title=f'{hours}h rule: {len(s)} of 37 meetings')
    ax.grid(alpha=.15)
fig.suptitle('Measurement and coverage change together')
fig.savefig(OUT / 'dispersion_comparison.png', dpi=160)
plt.show()


### How to interpret the result
Read coverage before coefficients. Compare fresh and authors' same-band rows to assess reconstruction changes; compare the authors' full measure with the earlier 37-meeting result to assess sample selection. A changed p-value alone cannot isolate a measurement problem, because sample size, membership, and the conditional definition all change.

A positive coefficient is an association, not evidence of causal effects or out-of-sample forecast skill. Even if it persists, this exercise cannot validate the original full variance, establish true probabilities, or establish that missing tails are harmless. Zero conditional dispersion is retained rather than silently dropped.

Sources: [author replication repository](https://github.com/jdkatz21/Prediction_Markets_Public), [daily distributions](https://kalshi-and-the-rise-of-macro-markets.s3.amazonaws.com/daily_distribution_data/daily_distributions_fed_levels.csv), [USMPD](https://www.frbsf.org/wp-content/uploads/USMPD.xlsx), [six-month benchmark](https://www.frbsf.org/wp-content/uploads/market-based-monetary-policy-uncertainty-data.xlsx), [historical trades API](https://docs.kalshi.com/api-reference/historical/get-historical-trades). Meeting inputs are the verified Step 7 snapshot; authors' probability bins are the Step 9 source snapshot. Trade summaries were retrieved October 19, 2026. The API historical cutoff was August 20, 2026; September queries use the current trades endpoint. Each summary records its endpoint. Refreshing these snapshots is a separate data-acquisition step, not part of a routine rerun.


In [ ]:
manifest = {'snapshot_sha256': EXPECTED_SHA256, 'rules': {'primary_age_hours':48, 'secondary_age_hours':24, 'each_implied_tail_max':.01, 'policy_grid_bp':25, 'simultaneous_trade_rule':'volume-weighted latest timestamp batch', 'monotonic_rule':'equal-weight decreasing isotonic'}, 'coverage':coverage.groupby('max_age_hours').eligible.sum().to_dict(), 'limitations':['source-screened candidate thresholds','conditional central-band variance','selected small samples','in-sample exploratory associations']}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
assert len(coverage) == 74
assert coverage.loc[coverage.eligible, 'central_implied_mass'].ge(.98-1e-10).all()
assert coverage.loc[coverage.eligible, 'fresh_central_sigma_bps'].ge(0).all()
print('Finished. Save this notebook back to GitHub with its outputs.')
